In [ ]:

# =========================== REPO PATHS ===========================
from pathlib import Path
def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "code" / "data").exists():
            return candidate
    if cwd.name == "code" and (cwd / "data").exists():
        return cwd.parent
    return cwd

PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / "code" / "data"

def _resolve_csv_path(filename: str) -> str:
    path = Path(filename)
    if path.exists():
        return str(path)
    for candidate in (
        DATA_DIR / path.name,
        PROJECT_ROOT / path.name,
        PROJECT_ROOT / "data" / path.name,
    ):
        if candidate.exists():
            return str(candidate)
    return str(DATA_DIR / path.name)

def _project_out(*parts: str) -> Path:
    return PROJECT_ROOT.joinpath(*parts)


import os, ast, random, inspect
from pathlib import Path
from typing import Dict, List
import numpy as np, pandas as pd, torch
import torch.utils.data as torchdata
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt, seaborn as sns
from transformers import AutoTokenizer, AutoModel, BertModel, AutoConfig
from IsoScore import IsoScore
from dadapy import Data
from skdim.id import MLE, MOM, TLE, CorrInt, FisherS, lPCA

# Bert POS

In [ ]:
import os, gc, ast, random, inspect
from pathlib import Path
from typing import Dict, List, Tuple, Callable

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel

HAS_DADAPY = False
try:
    from dadapy import Data  
    HAS_DADAPY = True
except Exception:
    pass

HAS_SKDIM = False
try:
    from skdim.id import (
        MOM, TLE, CorrInt, FisherS, lPCA,
        MLE, DANCo, ESS, MiND_ML, MADA, KNN
    )
    HAS_SKDIM = True
except Exception:
    pass

try:
    from IsoScore import IsoScore
    _HAS_ISOSCORE = True
except Exception:
    _HAS_ISOSCORE = False
    class _IsoScoreFallback:
        @staticmethod
        def IsoScore(X: np.ndarray) -> float:
            C = np.cov(X.T, ddof=0)
            ev = np.linalg.eigvalsh(C)
            if ev.mean() <= 0 or ev[-1] <= 0:
                return 0.0
            # mean/peak eigenvalue ratio in [0,1]; higher ≈ more isotropic
            return float(np.clip(ev.mean() / ev[-1], 0.0, 1.0))
    IsoScore = _IsoScoreFallback()

CSV_PATH   = _resolve_csv_path("en_ewt-ud-train_sentences.csv")
BASELINE   = "bert-base-uncased"
WORD_REP_MODE = "first"     
EXCLUDE_POS = {"X", "SYM", "PART", "INTJ"}
RAW_MAX_PER_POS = int(1e12)         





N_BOOTSTRAP_FAST   = 50         
N_BOOTSTRAP_HEAVY  = 200          

FAST_BS_MAX_SAMP_PER_POS  = int(1e12)   
HEAVY_BS_MAX_SAMP_PER_POS = 5000       
# GRIDE multi-scale max neighbor rank
DADAPY_GRID_RANGE_MAX = 64             # 32–128 is typical
RAND_SEED=42
PLOT_DIR     = _project_out("results_POS"); PLOT_DIR.mkdir(exist_ok=True, parents=True)
CSV_DIR      = _project_out("tables_POS") / "pos_bootstrap"; CSV_DIR.mkdir(exist_ok=True, parents=True)

# Throughput: start higher than 1 unless GPU is tiny
BATCH_SIZE = 1                         # try 8 → 16 → 32; back off if OOM

# Reproducibility & device
os.environ["TOKENIZERS_PARALLELISM"] = "true"
random.seed(RAND_SEED); np.random.seed(RAND_SEED); torch.manual_seed(RAND_SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda": torch.backends.cudnn.benchmark = True

# Seaborn style
sns.set_style("darkgrid")

plt.rcParams["figure.dpi"] = 120


# =============================== HELPERS ===============================
def _to_list(x):
    return ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x

def _center(X: np.ndarray) -> np.ndarray:
    return X - X.mean(0, keepdims=True)

def _eigvals_from_X(X: np.ndarray) -> np.ndarray:
    """Eigenvalues of covariance up to a constant via SVD of centered X (descending)."""
    Xc = _center(X.astype(np.float32, copy=False))
    try:
        _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        lam = (S**2).astype(np.float64)
        lam.sort()
        return lam[::-1]
    except Exception:
        return np.array([], dtype=np.float64)

def _jitter_unique(X: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Add tiny noise if there are duplicate rows (helps NN-based estimators)."""
    try:
        if np.unique(X, axis=0).shape[0] < X.shape[0]:
            X = X + np.random.normal(scale=eps, size=X.shape).astype(X.dtype)
    except Exception:
        pass
    return X

# ========= Per-subsample single-value compute functions (used inside bootstrap) =========
# --- Isotropy (fast) ---

def _fast_isoscore(X: np.ndarray) -> float:
    """Fast IsoScore from covariance eigenvalues; matches the package algorithm."""
    try:
        lam = _eigvals_from_X(X)
    except Exception:
        Xc = np.asarray(X, dtype=np.float32)
        if Xc.ndim != 2 or Xc.size == 0:
            return float("nan")
        Xc = Xc - Xc.mean(0, keepdims=True)
        try:
            _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        except Exception:
            return float("nan")
        lam = (S.astype(np.float64) ** 2)
        lam = np.sort(np.maximum(lam, 0.0))[::-1]

    lam = np.asarray(lam, dtype=np.float64)
    lam = lam[np.isfinite(lam) & (lam >= 0)]
    n = int(lam.size)
    if n < 2:
        return float("nan")
    norm = float(np.linalg.norm(lam))
    if norm <= 0:
        return 0.0

    root_n = np.sqrt(n)
    denom = np.sqrt(2.0 * (n - root_n))
    if denom <= 0:
        return float("nan")

    delta = float(np.linalg.norm((root_n * lam) / norm - 1.0) / denom)
    delta = float(np.clip(delta, 0.0, 1.0))
    phi = (n - (delta ** 2) * (n - root_n)) ** 2 / (n ** 2)
    iso = (n * phi - 1.0) / (n - 1.0)
    return float(np.clip(iso, 0.0, 1.0))


def _iso_once(X: np.ndarray) -> float:
    return float(_fast_isoscore(X))

def _spect_once(X: np.ndarray) -> float:
    ev = np.linalg.eigvalsh(np.cov(X.T, ddof=0))
    return float(ev[-1] / (ev.mean() + 1e-9))

def _rand_once(X: np.ndarray, K: int = 2000) -> float:
    n = X.shape[0]
    if n < 2: return np.nan
    rng = np.random.default_rng()
    K_eff = min(K, (n*(n-1))//2)
    i = rng.integers(0, n, size=K_eff)
    j = rng.integers(0, n, size=K_eff)
    same = i == j
    if same.any():
        j[same] = rng.integers(0, n, size=same.sum())
    A, B = X[i], X[j]
    num = np.sum(A*B, axis=1)
    den = (np.linalg.norm(A, axis=1)*np.linalg.norm(B, axis=1) + 1e-9)
    return float(np.mean(np.abs(num/den)))

def _sf_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    gm = np.exp(np.mean(np.log(lam + EPS)))
    am = float(lam.mean() + EPS)
    return float(gm / am)

def _pfI_once(X: np.ndarray) -> float:
    n, d = X.shape
    if n < 2: return np.nan
    rng = np.random.default_rng()
    U = rng.standard_normal((PFI_DIRS, d)).astype(np.float32)
    U /= np.linalg.norm(U, axis=1, keepdims=True) + 1e-9
    S = U @ X.T
    m = np.max(S, axis=1, keepdims=True)
    logZ = (m + np.log(np.sum(np.exp(S - m), axis=1, keepdims=True))).ravel()
    lo = np.percentile(logZ, PFI_Q_LO)
    hi = np.percentile(logZ, PFI_Q_HI)
    return float(np.exp(lo - hi))  # ≈ min Z / max Z (robust)

def _vmf_kappa_once(X: np.ndarray) -> float:
    if X.shape[0] < 2: return np.nan
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    R = np.linalg.norm(Xn.mean(axis=0))
    d = Xn.shape[1]
    if R < 1e-9: return 0.0
    # standard closed-form approximation
    return float(max(R * (d - R**2) / (1.0 - R**2 + 1e-9), 0.0))

# --- Linear ID (fast) ---
def _pcaXX_once(X: np.ndarray, var_ratio: float) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    c = np.cumsum(lam); thr = c[-1] * var_ratio
    return float(np.searchsorted(c, thr) + 1)

def _pca95_once(X: np.ndarray) -> float:
    return _pcaXX_once(X, 0.95)

def _pca99_once(X: np.ndarray) -> float:
    return _pcaXX_once(X, 0.99)

def _erank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    p = lam / (lam.sum() + EPS)
    H = -(p * np.log(p + EPS)).sum()
    return float(np.exp(H))

def _pr_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    s1 = lam.sum(); s2 = (lam**2).sum()
    return float((s1**2) / (s2 + EPS))

def _stable_rank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    return float(lam.sum() / (lam.max() + EPS))

# --- Non-linear (heavy) ---
def _dadapy_twonn_once(X: np.ndarray) -> float:
    if not HAS_DADAPY: return np.nan
    d = Data(coordinates=_jitter_unique(X))
    id_est, _, _ = d.compute_id_2NN()
    return float(id_est)

def _dadapy_gride_once(X: np.ndarray) -> float:
    if not HAS_DADAPY: return np.nan
    d = Data(coordinates=_jitter_unique(X))
    range_max = min(int(DADAPY_GRID_RANGE_MAX), X.shape[0] - 1)
    if range_max < 2:
        return float("nan")
    d.compute_distances(maxk=range_max)
    ids, _, _ = d.return_id_scaling_gride(range_max=range_max)
    return float(ids[-1])

def _skdim_factory(name: str):
    """Return a factory that builds a fresh skdim estimator each call, or None."""
    if not HAS_SKDIM: return None
    mapping = {
        "mom": MOM, "tle": TLE, "corrint": CorrInt, "fishers": FisherS,
        "lpca": lPCA, "lpca99": lPCA,
        "mle": MLE, "danco": DANCo,  "mind_ml": MiND_ML,
        "mada": MADA, "knn": KNN,
    }
    cls = mapping.get(name)
    if cls is None: return None

    def _builder():
        if name == "lpca":      # FO variant
            return cls(ver="FO")
        elif name == "lpca99":  # ratio (0.99) variant
            return cls(ver="ratio", alphaRatio=0.99)
        else:
            return cls()
    return _builder

def _skdim_once_builder(name: str) -> Callable[[np.ndarray], float] | None:
    build = _skdim_factory(name)
    if build is None: return None

    def _once(X: np.ndarray) -> float:
        est = build()
        est.fit(_jitter_unique(X))
        return float(getattr(est, "dimension_", np.nan))
    return _once


# =============================== DATA ===============================
def load_word_df(csv_path: str, exclude_pos: set[str] = EXCLUDE_POS):
    df = pd.read_csv(csv_path, usecols=["sentence_id","tokens","pos"])
    df["sentence_id"] = df["sentence_id"].astype(str)  # keep string IDs
    df.tokens = df.tokens.apply(_to_list); df.pos = df.pos.apply(_to_list)

    rows = []
    for sid, toks, poss in df[["sentence_id","tokens","pos"]].itertuples(index=False):
        for wid, (tok, p) in enumerate(zip(toks, poss)):
            if p not in exclude_pos:
                rows.append((sid, wid, p, tok))
    word_df = pd.DataFrame(rows, columns=["sentence_id","word_id","pos","word"])
    return df, word_df

def sample_raw(word_df: pd.DataFrame, per_pos_cap: int = RAW_MAX_PER_POS) -> pd.DataFrame:
    """Per-POS cap without frequency matching."""
    picks = []
    for p, sub in word_df.groupby("pos", sort=False):
        n = min(len(sub), per_pos_cap)
        picks.append(sub.sample(n, random_state=RAND_SEED, replace=False))
    return pd.concat(picks, ignore_index=True)


def make_class_palette(classes: List[str]) -> Dict[str, Tuple[float, float, float]]:
    """
    Return a deterministic mapping {class -> RGB tuple} with as many distinct
    qualitative colors as needed. Up to ~60 unique colors without reuse.
    """
    # Try three matplotlib tab palettes first (20 + 20 + 20)
    base_colors: List[Tuple[float, float, float]] = []
    for name in ("tab20", "tab20b", "tab20c"):
        try:
            base_colors.extend(sns.color_palette(name, 20))
        except Exception:
            pass

    # If classes exceed our pool, fall back to evenly spaced hues
    if len(base_colors) < len(classes):
        base_colors = list(sns.color_palette("husl", len(classes)))  # evenly spaced hues

    # Deterministic order (sorted) -> stable color assignment
    ordered = list(sorted(classes))
    return {cls: base_colors[i % len(base_colors)] for i, cls in enumerate(ordered)}

# =============================== EMBEDDING ===============================
def embed_subset(df_all_sentences: pd.DataFrame,
                 subset_df: pd.DataFrame,
                 baseline: str = BASELINE,
                 word_rep_mode: str = WORD_REP_MODE,
                 batch_size: int = BATCH_SIZE) -> Tuple[np.ndarray, np.ndarray]:
    """Return reps (L,N,D) and filled mask (N,) for the selected tokens."""
    df_all_sentences["sentence_id"] = df_all_sentences["sentence_id"].astype(str)
    subset_df["sentence_id"] = subset_df["sentence_id"].astype(str)

    # sid -> list[(global_idx, word_id)]
    by_sid: Dict[str, List[Tuple[int,int]]] = {}
    for gidx, (sid, wid) in enumerate(subset_df[["sentence_id","word_id"]].itertuples(index=False)):
        by_sid.setdefault(str(sid), []).append((gidx, int(wid)))

    # Materialize in the exact order we will batch
    sids = list(by_sid.keys())
    df_sel = (df_all_sentences[df_all_sentences.sentence_id.isin(sids)]
              .drop_duplicates("sentence_id")
              .set_index("sentence_id")
              .loc[sids])

    tokzr = AutoTokenizer.from_pretrained(baseline, use_fast=True, add_prefix_space=True)
    enc_kwargs = dict(is_split_into_words=True, return_tensors="pt", padding=True)
    if "add_prefix_space" in inspect.signature(tokzr.__call__).parameters:
        enc_kwargs["add_prefix_space"] = True

    model = AutoModel.from_pretrained(baseline, output_hidden_states=True).eval().to(device)
    if device == "cuda":
        model.half()

    L = model.config.num_hidden_layers + 1
    D = model.config.hidden_size
    N = len(subset_df)

    reps   = np.zeros((L, N, D), np.float16)
    filled = np.zeros(N, dtype=bool)

    with torch.no_grad(), torch.cuda.amp.autocast(device == "cuda"):
        for start in tqdm(range(0, len(sids), batch_size), desc=f"{baseline} (embed subset)"):
            batch_ids    = sids[start : start + batch_size]
            batch_tokens = df_sel.loc[batch_ids, "tokens"].tolist()

            enc_be = tokzr(batch_tokens, **enc_kwargs)
            enc_t  = {k: v.to(device) for k, v in enc_be.items()}
            out = model(**enc_t)
            h = torch.stack(out.hidden_states).detach().cpu().numpy().astype(np.float32)  # (L,B,T,D)

            for b, sid in enumerate(batch_ids):
                # word_id -> token positions for this item
                mp = {}
                for tidx, wid in enumerate(enc_be.word_ids(b)):
                    if wid is not None:
                        mp.setdefault(int(wid), []).append(int(tidx))

                for gidx, wid in by_sid.get(sid, []):
                    toks = mp.get(wid)
                    if not toks: continue
                    if word_rep_mode == "first":
                        vec = h[:, b, toks[0], :]
                    else:
                        vec = h[:, b, toks, :].mean(axis=1)
                    reps[:, gidx, :] = vec.astype(np.float16, copy=False)
                    filled[gidx] = True

            # free batch buffers
            del enc_be, enc_t, out, h
            if device == "cuda": torch.cuda.empty_cache()

    missing = int((~filled).sum())
    if missing:
        print(f"⚠ Missing vectors for {missing} of {N} sampled words")
    del model; gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    return reps, filled


# =============================== BOOTSTRAP CORE ===============================
def _bs_layer_loop(rep_sub: np.ndarray, M: int, n_reps: int, compute_once: Callable[[np.ndarray], float]):
    """Bootstrap: sample M with replacement and apply compute_once(X_layer) -> scalar for each layer."""
    L, N, D = rep_sub.shape
    rng = np.random.default_rng(RAND_SEED)
    A = np.full((n_reps, L), np.nan, np.float32)
    for r in range(n_reps):
        idx = rng.integers(0, N, size=M)
        for l in range(L):
            X = rep_sub[l, idx].astype(np.float32, copy=False)
            try:
                A[r, l] = float(compute_once(X))
            except Exception:
                A[r, l] = np.nan
    mu = np.nanmean(A, axis=0).astype(np.float32)
    lo = np.nanpercentile(A, 2.5, axis=0).astype(np.float32)
    hi = np.nanpercentile(A, 97.5, axis=0).astype(np.float32)
    return mu, lo, hi

# fast metric registry (name -> callable(X)->float)
FAST_ONCE: Dict[str, Callable[[np.ndarray], float]] = {
    "iso": _iso_once,
    "spect": _spect_once,
    "rand": _rand_once,
    "sf": _sf_once,
    "vmf_kappa": _vmf_kappa_once,
    "erank": _erank_once,
    "pr": _pr_once,
    "stable_rank": _stable_rank_once,
}

# heavy metric registry
HEAVY_ONCE: Dict[str, Callable[[np.ndarray], float] | None] = {
    "twonn": _dadapy_twonn_once,
    "gride": _dadapy_gride_once,
    "mom":   _skdim_once_builder("mom"),
    "tle":   _skdim_once_builder("tle"),
    "corrint": _skdim_once_builder("corrint"),
    "fishers": _skdim_once_builder("fishers"),
    "lpca":  _skdim_once_builder("lpca"),
    "lpca95": _skdim_once_builder("lpca95"),
    "lpca99": _skdim_once_builder("lpca99"),
    "mle":   _skdim_once_builder("mle"),
    "mada":  _skdim_once_builder("mada"),
    "knn":   _skdim_once_builder("knn"),
}

LABELS = {
    # Isotropy
    "iso":"IsoScore","spect":"Spectral Ratio","rand":"RandCos |μ|",
    "sf":"Spectral Flatness","vmf_kappa":"vMF κ",
    # Linear ID
    "erank":"Effective Rank","pr":"Participation Ratio","stable_rank":"Stable Rank",
    "lpca95":"lPCA95","lpca99":"lPCA99","lpca":"lPCA FO",
    # Non-linear
    "twonn":"TwoNN ID","gride":"GRIDE",
    "mom":"MOM","tle":"TLE","corrint":"CorrInt",
    "fishers":"FisherS",
    "mle":"MLE","mada":"MADA","knn":"KNN",
}

# Choose plotting order
PLOT_ORDER = (
    "iso","sf","vmf_kappa","spect","rand",
    "erank","pr","stable_rank","lpca95","lpca99","lpca",
    "twonn","gride","mom","tle","corrint","fishers",
    "mle","mada","knn"
)

# metrics you want to compute (you can prune this list to reduce runtime)
#ALL_METRICS = list(PLOT_ORDER)
ALL_METRICS = ["iso"]


# =============================== SAVE / PLOT ===============================
def save_metric_csv_all_pos(metric: str,
                            pos_to_stats: Dict[str, Dict[str, np.ndarray]],
                            layers: np.ndarray,
                            baseline: str,
                            subset_name: str = "raw"):
    rows = []
    for p, stats in pos_to_stats.items():
        mu, lo, hi, n = stats["mean"], stats.get("lo"), stats.get("hi"), stats.get("n", np.nan)
        for l, val in enumerate(mu):
            rows.append({
                "subset": subset_name, "model": baseline, "feature": "pos",
                "class": p, "metric": metric, "layer": int(layers[l]),
                "mean": float(val) if np.isfinite(val) else np.nan,
                "ci_low": float(lo[l]) if isinstance(lo, np.ndarray) and np.isfinite(lo[l]) else np.nan,
                "ci_high": float(hi[l]) if isinstance(hi, np.ndarray) and np.isfinite(hi[l]) else np.nan,
                "n_tokens": int(stats.get("n", 0)), "word_rep_mode": WORD_REP_MODE,
                "source_csv": Path(CSV_PATH).name,
            })
    df = pd.DataFrame(rows)
    out = CSV_DIR / f"pos_{subset_name}_{metric}_{baseline}.csv"
    df.to_csv(out, index=False)

def plot_metric_with_ci(pos_to_stats: Dict[str, Dict[str, np.ndarray]],
                        layers: np.ndarray, metric: str, title: str, out_path: Path,
                        palette: Dict[str, Tuple[float, float, float]] | None = None):
    plt.figure(figsize=(9, 5))
    for p, stats in pos_to_stats.items():
        mu, lo, hi = stats["mean"], stats.get("lo"), stats.get("hi")
        if mu is None or np.all(np.isnan(mu)): 
            continue
        color = palette.get(p) if isinstance(palette, dict) else None
        plt.plot(layers, mu, label=p, lw=1.8, color=color)
        if isinstance(lo, np.ndarray) and isinstance(hi, np.ndarray) and not np.all(np.isnan(lo)):
            plt.fill_between(layers, lo, hi, alpha=0.15, color=color)
    plt.xlabel("Layer"); plt.ylabel(LABELS.get(metric, metric.upper())); plt.title(title)

    # Make the legend compact if there are many classes
    n_classes = len(pos_to_stats)
    ncol = 3 if n_classes > 12 else 2
    plt.legend(ncol=ncol, fontsize="small", title="POS", frameon=False)

    plt.tight_layout(); plt.savefig(out_path, dpi=220); plt.close()



# =============================== DRIVER ===============================
def run_pos_pipeline():
    # 1) Load
    df_all, word_df = load_word_df(CSV_PATH, EXCLUDE_POS)
    POS_TAGS = sorted(word_df.pos.unique())
    palette = make_class_palette(POS_TAGS)
    print(f"✓ corpus ready — {len(word_df):,} tokens across {len(POS_TAGS)} POS")
    print(f"• DADApy: {'available' if HAS_DADAPY else 'missing'}  • scikit-dimension: {'available' if HAS_SKDIM else 'missing'}")

    # 2) Raw sampling only (no frequency match)
    raw_df = sample_raw(word_df, RAW_MAX_PER_POS)
    print("Sample sizes per POS (raw cap):")
    print(raw_df.pos.value_counts().to_dict())

    # 3) Embed once
    reps, filled = embed_subset(df_all, raw_df, BASELINE, WORD_REP_MODE, BATCH_SIZE)
    raw_df = raw_df.reset_index(drop=True).loc[filled].reset_index(drop=True)
    pos_arr = raw_df.pos.values
    L = reps.shape[0]; layers = np.arange(L)
    print(f"✓ embedded {len(raw_df):,} tokens  • layers={L}")

    # 4) Metric-by-metric loop (incremental outputs)
    for metric in ALL_METRICS:
        print(f"\n→ Computing metric: {metric} …")

        # Decide registry & bootstrap settings
        if metric in FAST_ONCE:
            compute_once = FAST_ONCE[metric]
            n_bs = N_BOOTSTRAP_FAST
            Mcap = FAST_BS_MAX_SAMP_PER_POS
        else:
            compute_once = HEAVY_ONCE.get(metric)
            n_bs = N_BOOTSTRAP_HEAVY
            Mcap = HEAVY_BS_MAX_SAMP_PER_POS

        if compute_once is None:
            print(f"  (skipping {metric}: estimator unavailable)")
            continue

        # Per-POS bootstrap
        metric_results: Dict[str, Dict[str, np.ndarray]] = {}
        for p in POS_TAGS:
            idx = np.where(pos_arr == p)[0]
            if idx.size < 3:
                continue
            sub = reps[:, idx]  # (L, n_p, D)
            Np = sub.shape[1]
            M = min(Mcap, Np)

            mu, lo, hi = _bs_layer_loop(sub, M, n_bs, compute_once)
            metric_results[p] = {"mean": mu, "lo": lo, "hi": hi, "n": int(Np)}

        # Save + plot immediately for this metric
        save_metric_csv_all_pos(metric, metric_results, layers, BASELINE, subset_name="raw")
        plot_metric_with_ci(metric_results, layers, metric,
                    title=f"{LABELS.get(metric, metric.upper())} • {BASELINE}",
                    out_path=PLOT_DIR / f"raw_{metric}_{BASELINE}.png",
                    palette=palette)

        print(f"  ✓ saved: CSV= tables/pos_bootstrap/pos_raw_{metric}_{BASELINE}.csv  "
              f"plot= AO_POS/raw_{metric}_{BASELINE}.png")

        # light cleanup for safety
        del metric_results; gc.collect()
        if device == "cuda": torch.cuda.empty_cache()

    # Cleanup
    del reps; gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    print("\n✓ done (incremental outputs produced per metric).")


if __name__ == "__main__":
    run_pos_pipeline()


In [ ]:
import os, gc, ast, random, inspect
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import matplotlib.pyplot as plt  # only for color palettes

CSV_PATH   = _resolve_csv_path("en_ewt-ud-train_sentences.csv")   

BASELINE      = "gpt2"          
WORD_REP_MODE = "last"                        

# Plotting / sampling
PCA_PER_CLASS_MAX_POINTS = 3000              

# Throughput + device
BATCH_SIZE = 2
os.environ["TOKENIZERS_PARALLELISM"] = "true"
random.seed(RAND_SEED); np.random.seed(RAND_SEED); torch.manual_seed(RAND_SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.backends.cudnn.benchmark = True

# Output
OUT_DIR = _project_out("pca3d_pos_classes"); OUT_DIR.mkdir(parents=True, exist_ok=True)
HTML_OUT = OUT_DIR / f"{BASELINE.replace('/','_')}_pca3d_pos_classes.html"

# =============================== HELPERS ===============================
def _to_list(x):
    return ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x

def _num_hidden_layers(model) -> int:
    n = getattr(model.config, "num_hidden_layers", None)
    if n is None: n = getattr(model.config, "n_layer", None)
    if n is None: raise ValueError("Cannot determine number of hidden layers")
    return int(n)

def _hidden_size(model) -> int:
    d = getattr(model.config, "hidden_size", None)
    if d is None: d = getattr(model.config, "n_embd", None)
    if d is None: raise ValueError("Cannot determine hidden size")
    return int(d)

def _load_tok_and_model(model_id: str):
    """
    Robust loader (BERT/GPT‑2):
      - fast tokenizer for .word_ids()
      - right padding
      - set PAD=EOS for GPT‑like models
    """
    tried = []
    def _try(mid: str):
        tok = AutoTokenizer.from_pretrained(mid, use_fast=True, add_prefix_space=True)
        if getattr(tok, "padding_side", None) != "right":
            tok.padding_side = "right"
        if tok.pad_token is None and getattr(tok, "eos_token", None) is not None:
            tok.pad_token = tok.eos_token
        mdl = AutoModel.from_pretrained(mid, output_hidden_states=True)
        if getattr(mdl.config, "pad_token_id", None) is None and tok.pad_token_id is not None:
            mdl.config.pad_token_id = tok.pad_token_id
        return tok, mdl

    order = [model_id]
    if model_id.lower() in {"gpt2", "gpt-2"}:
        order += ["openai-community/gpt2", "gpt2"]  # tolerate both namespaces

    last_err = None
    for mid in order:
        try:
            tok, mdl = _try(mid)
            mdl = mdl.eval().to(device)
            if device == "cuda": mdl.half()
            return tok, mdl, mid
        except Exception as e:
            tried.append((mid, repr(e))); last_err = e

    raise RuntimeError("Could not load tokenizer/model. Attempts:\n" +
                       "\n".join(f" - {m}: {err}" for m, err in tried)) from last_err

# =============================== DATA (POS classes) ===============================
def load_pos_tokens(csv_path: str, exclude: set[str] | None = None):
    """
    Expects per-sentence: tokens (list[str]) and pos (list[str]).
    Returns:
      df_sent: sentence_id, tokens
      df_tok : per-token rows with columns [sentence_id, word_id, pos, word]
    """
    df = pd.read_csv(csv_path, usecols=["sentence_id","tokens","pos"])
    df["sentence_id"] = df["sentence_id"].astype(str)
    df.tokens = df.tokens.apply(_to_list)
    df.pos    = df.pos.apply(_to_list)
    rows = []
    for sid, toks, poss in df[["sentence_id","tokens","pos"]].itertuples(index=False):
        L = min(len(toks), len(poss))
        for wid in range(L):
            p = str(poss[wid])
            if exclude and p in exclude:
                continue
            rows.append((sid, wid, p, str(toks[wid])))
    df_tok  = pd.DataFrame(rows, columns=["sentence_id","word_id","pos","word"])
    df_sent = df[["sentence_id","tokens"]].drop_duplicates("sentence_id")
    if df_tok.empty:
        raise ValueError("No token rows constructed—check POS column content.")
    return df_sent, df_tok

# =============================== EMBEDDING ===============================
def embed_subset(df_sent: pd.DataFrame,
                 subset_df: pd.DataFrame,
                 baseline: str = BASELINE,
                 word_rep_mode: str = WORD_REP_MODE,
                 batch_size: int = BATCH_SIZE) -> Tuple[np.ndarray, np.ndarray, str]:
    """
    Returns:
      reps   (L, N, D)
      filled (N,)
      model_tag (resolved id)
    """
    df_sent["sentence_id"]   = df_sent["sentence_id"].astype(str)
    subset_df["sentence_id"] = subset_df["sentence_id"].astype(str)

    # sid -> list[(global_idx, word_id)]
    by_sid: Dict[str, List[Tuple[int,int]]] = {}
    for gidx, (sid, wid) in enumerate(subset_df[["sentence_id","word_id"]].itertuples(index=False)):
        by_sid.setdefault(str(sid), []).append((gidx, int(wid)))

    sids = list(by_sid.keys())
    df_sel = (df_sent[df_sent.sentence_id.isin(sids)]
              .drop_duplicates("sentence_id")
              .set_index("sentence_id")
              .loc[sids])

    tokzr, model, model_id = _load_tok_and_model(baseline)

    enc_kwargs = dict(is_split_into_words=True, return_tensors="pt", padding=True, truncation=True)
    if "add_prefix_space" in inspect.signature(tokzr.__call__).parameters:
        enc_kwargs["add_prefix_space"] = True

    L = _num_hidden_layers(model) + 1
    D = _hidden_size(model)
    N = len(subset_df)

    reps   = np.zeros((L, N, D), np.float16)
    filled = np.zeros(N, dtype=bool)

    with torch.no_grad(), torch.cuda.amp.autocast(device == "cuda"):
        for start in tqdm(range(0, len(sids), batch_size), desc=f"{model_id} (embed subset)"):
            batch_ids    = sids[start : start + batch_size]
            batch_tokens = df_sel.loc[batch_ids, "tokens"].tolist()

            enc_be = tokzr(batch_tokens, **enc_kwargs)
            enc_t  = {k: v.to(device) for k, v in enc_be.items()}
            out = model(**enc_t)
            h = torch.stack(out.hidden_states).detach().cpu().numpy().astype(np.float32)  # (L,B,T,D)

            for b, sid in enumerate(batch_ids):
                # map word_id -> token positions
                mp: Dict[int, List[int]] = {}
                wids = enc_be.word_ids(b)
                if wids is None:
                    raise RuntimeError("Fast tokenizer required (word_ids() unavailable).")
                for tidx, wid in enumerate(wids):
                    if wid is not None:
                        mp.setdefault(int(wid), []).append(int(tidx))

                for gidx, wid in by_sid.get(sid, []):
                    toks = mp.get(wid)
                    if not toks: continue
                    if word_rep_mode == "first":
                        vec = h[:, b, toks[0], :]
                    elif word_rep_mode == "last":
                        vec = h[:, b, toks[-1], :]
                    elif word_rep_mode == "mean":
                        vec = h[:, b, toks, :].mean(axis=1)
                    else:
                        raise ValueError("WORD_REP_MODE must be {'first','last','mean'}")
                    reps[:, gidx, :] = vec.astype(np.float16, copy=False)
                    filled[gidx] = True

            del enc_be, enc_t, out, h
            if device == "cuda": torch.cuda.empty_cache()

    missing = int((~filled).sum())
    if missing:
        print(f"⚠ Missing vectors for {missing} of {N} tokens")
    del model; gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    return reps, filled, model_id

# =============================== COLORS ===============================
def _class_palette(classes: List[str]) -> Dict[str, str]:
    """
    Distinct hex colors for many POS tags: use tab20+tab20b+tab20c (~60 colors), then hsv fallback.
    """
    colors = []
    for name in ("tab20", "tab20b", "tab20c"):
        try:
            colors.extend(plt.get_cmap(name).colors)
        except Exception:
            pass
    if len(colors) < len(classes):
        hsv = plt.get_cmap("hsv")
        colors = [hsv(i/len(classes)) for i in range(len(classes))]
    to_hex = lambda rgb: "#{:02x}{:02x}{:02x}".format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
    ordered = list(sorted(classes))  # alphabetical for stability
    return {cls: to_hex(colors[i % len(colors)]) for i, cls in enumerate(ordered)}

# =============================== PCA + PLOTLY BY CLASS ===============================
def pca3d_layers_by_class(reps: np.ndarray,
                          words: List[str],
                          class_arr: np.ndarray,
                          classes: List[str],
                          model_tag: str,
                          html_out: Path):
    """
    reps: (L, N, D)
    words: list[str] length N (hover text)
    class_arr: array length N with POS labels (strings)
    classes: list of unique labels
    """
    L, N, D = reps.shape
    classes_sorted = list(sorted(classes))
    palette = _class_palette(classes_sorted)

    # Build consistent subset across layers: sample up to cap per class
    rng = np.random.default_rng(RAND_SEED)
    sel_idx: List[int] = []
    for c in classes_sorted:
        idx_c = np.where(class_arr == c)[0]
        if PCA_PER_CLASS_MAX_POINTS is not None and len(idx_c) > PCA_PER_CLASS_MAX_POINTS:
            idx_c = rng.choice(idx_c, size=PCA_PER_CLASS_MAX_POINTS, replace=False)
        sel_idx.extend(idx_c.tolist())
    sel_idx = np.array(sel_idx, dtype=np.int64)
    if sel_idx.size == 0:
        raise ValueError("No points selected for PCA plotting (check POS tags and caps).")

    # Per-class local positions within sel_idx
    cls_pos: Dict[str, np.ndarray] = {c: np.where(class_arr[sel_idx] == c)[0] for c in classes_sorted}

    reps_sel = reps[:, sel_idx, :].astype(np.float32, copy=False)
    words_sel = [words[i] for i in sel_idx]

    # PCA per layer
    Y_layers: List[np.ndarray] = []
    for l in range(L):
        X = reps_sel[l]  # (n_sel, D)
        Xc = X - X.mean(0, keepdims=True)
        pca = PCA(n_components=3, random_state=RAND_SEED)
        Y = pca.fit_transform(Xc)  # (n_sel, 3)
        Y_layers.append(Y)

    # Build traces: (layer, class) grid
    traces = []
    for l in range(L):
        Y = Y_layers[l]
        for c in classes_sorted:
            pos = cls_pos[c]
            if pos.size == 0:
                continue
            # customdata for stable hover: [class, layer]
            custom = np.column_stack([np.full(pos.size, c, dtype=object),
                                      np.full(pos.size, l, dtype=int)])
            traces.append(
                go.Scatter3d(
                    x=Y[pos, 0], y=Y[pos, 1], z=Y[pos, 2],
                    mode="markers",
                    marker=dict(size=2, opacity=0.75, color=palette[c]),
                    text=[words_sel[i] for i in pos],
                    customdata=custom,
                    hovertemplate=(
                        "<b>%{text}</b>"
                        "<br>POS=%{customdata[0]} • layer=%{customdata[1]}"
                        "<br>x=%{x:.3f}<br>y=%{y:.3f}<br>z=%{z:.3f}"
                        "<extra></extra>"
                    ),
                    name=f"{c}",
                    visible=(l == 0),
                    showlegend=(l == 0)  # legend only once
                )
            )

    # Slider toggles visibility of all class traces for a given layer
    traces_per_layer = len(traces) // L if L > 0 else 0
    steps = []
    for l in range(L):
        vis = [False]*len(traces)
        start = l * traces_per_layer
        for k in range(traces_per_layer):
            if start + k < len(traces):
                vis[start + k] = True
        steps.append(dict(
            method="update",
            args=[{"visible": vis},
                  {"title": f"{model_tag} • PCA 3D by POS • layer {l} (drag to rotate)"}],
            label=str(l),
        ))

    sliders = [dict(
        active=0,
        steps=steps,
        currentvalue={"prefix": "Layer: "},
        pad={"t": 10}
    )]

    layout = go.Layout(
        title=f"{model_tag} • PCA 3D by POS • layer 0 (drag to rotate)",
        scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3", aspectmode="data"),
        margin=dict(l=0, r=0, b=0, t=40),
        sliders=sliders,
        showlegend=True
    )

    fig = go.Figure(data=traces, layout=layout)
    fig.write_html(str(html_out), include_plotlyjs="cdn")
    print("✓ Saved interactive HTML to:", html_out)

# =============================== DRIVER ===============================
def run_pca3d_pos_classes():
    # 1) Load POS tokens
    #    (Optionally exclude certain POS by passing exclude={"X","SYM","PART","INTJ"} etc.)
    df_sent, df_tok = load_pos_tokens(CSV_PATH, exclude=None)
    pos_tags = sorted(df_tok.pos.unique().tolist())
    print(f"✓ corpus ready — {len(df_tok):,} tokens across POS={pos_tags}")

    # 2) (Optional) cap per POS for speed happens inside the PCA function; here we keep all
    subset_df = df_tok[["sentence_id","word_id","pos","word"]].copy()

    # 3) Embed once
    reps, filled, resolved_model = embed_subset(df_sent, subset_df, BASELINE, WORD_REP_MODE, BATCH_SIZE)
    subset_df = subset_df.reset_index(drop=True).loc[filled].reset_index(drop=True)

    # 4) Collect hover text + labels
    words = subset_df["word"].astype(str).tolist()
    cls_arr = subset_df["pos"].astype(str).values
    classes = sorted(subset_df["pos"].unique().tolist())

    # 5) PCA+Plotly
    pca3d_layers_by_class(reps, words, cls_arr, classes, model_tag=resolved_model, html_out=HTML_OUT)

    # Cleanup
    del reps; gc.collect()
    if device == "cuda": torch.cuda.empty_cache()

if __name__ == "__main__":
    run_pca3d_pos_classes()


In [ ]:
import os, gc, ast, random, inspect
from pathlib import Path
from typing import Dict, List, Tuple, Callable

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel

# ============== Optional deps (gracefully skipped if not installed) ==============
HAS_DADAPY = False
try:
    from dadapy import Data  # DADApy ID estimators (TwoNN, GRIDE)
    HAS_DADAPY = True
except Exception:
    pass

HAS_SKDIM = False
try:
    from skdim.id import (
        MOM, TLE, CorrInt, FisherS, lPCA,
        MLE, DANCo, ESS, MiND_ML, MADA, KNN
    )
    HAS_SKDIM = True
except Exception:
    pass

# IsoScore: use library if available, else a simple monotone fallback
try:
    from IsoScore import IsoScore
    _HAS_ISOSCORE = True
except Exception:
    _HAS_ISOSCORE = False
    class _IsoScoreFallback:
        @staticmethod
        def IsoScore(X: np.ndarray) -> float:
            C = np.cov(X.T, ddof=0)
            ev = np.linalg.eigvalsh(C)
            if ev.mean() <= 0 or ev[-1] <= 0:
                return 0.0
            # mean/peak eigenvalue ratio in [0,1]; higher ≈ more isotropic
            return float(np.clip(ev.mean() / (ev[-1] + 1e-12), 0.0, 1.0))
    IsoScore = _IsoScoreFallback()

# =============================== CONFIG ===============================
CSV_PATH        = _resolve_csv_path("en_ewt-ud-train_sentences.csv")
BASELINE        = "gpt2"                 # ← GPT‑2
WORD_REP_MODE   = "last"                 # ← {"last","mean"} word representation
EXCLUDE_POS     = {"X", "SYM", "PART", "INTJ"}

# Sampling cap per POS for plotting (increase if you want)

RAW_MAX_PER_POS = int(1e12)         # effectively no cap

# Bootstrap replicates
N_BOOTSTRAP_FAST   = 50         # good CIs without going overboard
N_BOOTSTRAP_HEAVY  = 200            # keep heavy bootstrap moderate

# Per-replicate sample size M (min(cap, N_pos))
FAST_BS_MAX_SAMP_PER_POS  = int(1e12)   # => M = N_pos (the classic bootstrap uses M=N)
HEAVY_BS_MAX_SAMP_PER_POS = 5000        # 1000–5000 is a practical range for TwoNN/GRIDE/skdim

DADAPY_GRID_RANGE_MAX     = 64

# Runtime / output
BATCH_SIZE   = 8
RAND_SEED    = 42
PLOT_DIR     = _project_out("AO_POS"); PLOT_DIR.mkdir(exist_ok=True, parents=True)
CSV_DIR      = _project_out("tables") / "pos_bootstrap"; CSV_DIR.mkdir(exist_ok=True, parents=True)

# Isotropy extras
PFI_DIRS = 256
PFI_Q_LO = 5.0
PFI_Q_HI = 95.0
EPS      = 1e-12

# Repro & device
os.environ["TOKENIZERS_PARALLELISM"] = "true"
random.seed(RAND_SEED); np.random.seed(RAND_SEED); torch.manual_seed(RAND_SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda": torch.backends.cudnn.benchmark = True

# Seaborn style
sns.set_style("darkgrid")
plt.rcParams["figure.dpi"] = 120

# =============================== HELPERS ===============================
def _to_list(x):
    return ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x

def _center(X: np.ndarray) -> np.ndarray:
    return X - X.mean(0, keepdims=True)

def _eigvals_from_X(X: np.ndarray) -> np.ndarray:
    """Eigenvalues of covariance up to a constant via SVD of centered X (descending)."""
    Xc = _center(X.astype(np.float32, copy=False))
    try:
        _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        lam = (S**2).astype(np.float64)
        lam.sort()
        return lam[::-1]
    except Exception:
        return np.array([], dtype=np.float64)

def _jitter_unique(X: np.ndarray, eps: float = 1e-7) -> np.ndarray:
    """Add tiny noise if there are duplicate rows (helps NN-based estimators)."""
    try:
        if np.unique(X, axis=0).shape[0] < X.shape[0]:
            X = X + np.random.normal(scale=eps, size=X.shape).astype(X.dtype)
    except Exception:
        pass
    return X

# ========= Single-shot compute functions used inside bootstrap =========
# --- Isotropy (fast) ---

def _fast_isoscore(X: np.ndarray) -> float:
    """Fast IsoScore from covariance eigenvalues; matches the package algorithm."""
    try:
        lam = _eigvals_from_X(X)
    except Exception:
        Xc = np.asarray(X, dtype=np.float32)
        if Xc.ndim != 2 or Xc.size == 0:
            return float("nan")
        Xc = Xc - Xc.mean(0, keepdims=True)
        try:
            _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        except Exception:
            return float("nan")
        lam = (S.astype(np.float64) ** 2)
        lam = np.sort(np.maximum(lam, 0.0))[::-1]

    lam = np.asarray(lam, dtype=np.float64)
    lam = lam[np.isfinite(lam) & (lam >= 0)]
    n = int(lam.size)
    if n < 2:
        return float("nan")
    norm = float(np.linalg.norm(lam))
    if norm <= 0:
        return 0.0

    root_n = np.sqrt(n)
    denom = np.sqrt(2.0 * (n - root_n))
    if denom <= 0:
        return float("nan")

    delta = float(np.linalg.norm((root_n * lam) / norm - 1.0) / denom)
    delta = float(np.clip(delta, 0.0, 1.0))
    phi = (n - (delta ** 2) * (n - root_n)) ** 2 / (n ** 2)
    iso = (n * phi - 1.0) / (n - 1.0)
    return float(np.clip(iso, 0.0, 1.0))


def _iso_once(X: np.ndarray) -> float:
    return float(_fast_isoscore(X))

def _spect_once(X: np.ndarray) -> float:
    ev = np.linalg.eigvalsh(np.cov(X.T, ddof=0))
    return float(ev[-1] / (ev.mean() + 1e-9))

def _rand_once(X: np.ndarray, K: int = 2000) -> float:
    n = X.shape[0]
    if n < 2: return np.nan
    rng = np.random.default_rng()
    K_eff = min(K, (n*(n-1))//2)
    i = rng.integers(0, n, size=K_eff)
    j = rng.integers(0, n, size=K_eff)
    same = i == j
    if same.any():
        j[same] = rng.integers(0, n, size=same.sum())
    A, B = X[i], X[j]
    num = np.sum(A*B, axis=1)
    den = (np.linalg.norm(A, axis=1)*np.linalg.norm(B, axis=1) + 1e-9)
    return float(np.mean(np.abs(num/den)))

def _sf_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    gm = np.exp(np.mean(np.log(lam + EPS)))
    am = float(lam.mean() + EPS)
    return float(gm / am)

def _pfI_once(X: np.ndarray) -> float:
    n, d = X.shape
    if n < 2: return np.nan
    rng = np.random.default_rng()
    U = rng.standard_normal((PFI_DIRS, d)).astype(np.float32)
    U /= np.linalg.norm(U, axis=1, keepdims=True) + 1e-9
    S = U @ X.T
    m = np.max(S, axis=1, keepdims=True)
    logZ = (m + np.log(np.sum(np.exp(S - m), axis=1, keepdims=True))).ravel()
    lo = np.percentile(logZ, PFI_Q_LO)
    hi = np.percentile(logZ, PFI_Q_HI)
    return float(np.exp(lo - hi))  # ≈ min Z / max Z (robust)

def _vmf_kappa_once(X: np.ndarray) -> float:
    if X.shape[0] < 2: return np.nan
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    R = np.linalg.norm(Xn.mean(axis=0))
    d = Xn.shape[1]
    if R < 1e-9: return 0.0
    return float(max(R * (d - R**2) / (1.0 - R**2 + 1e-9), 0.0))

# --- Linear ID (fast) ---
def _pcaXX_once(X: np.ndarray, var_ratio: float) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    c = np.cumsum(lam); thr = c[-1] * var_ratio
    return float(np.searchsorted(c, thr) + 1)

def _pca95_once(X: np.ndarray) -> float:
    return _pcaXX_once(X, 0.95)

def _pca99_once(X: np.ndarray) -> float:
    return _pcaXX_once(X, 0.99)

def _erank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    p = lam / (lam.sum() + EPS)
    H = -(p * np.log(p + EPS)).sum()
    return float(np.exp(H))

def _pr_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    s1 = lam.sum(); s2 = (lam**2).sum()
    return float((s1**2) / (s2 + EPS))

def _stable_rank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    return float(lam.sum() / (lam.max() + EPS))

# --- Non-linear (heavy) ---
def _dadapy_twonn_once(X: np.ndarray) -> float:
    if not HAS_DADAPY: return np.nan
    d = Data(coordinates=_jitter_unique(X))
    id_est, _, _ = d.compute_id_2NN()
    return float(id_est)

def _dadapy_gride_once(X: np.ndarray) -> float:
    if not HAS_DADAPY: return np.nan
    d = Data(coordinates=_jitter_unique(X))
    range_max = min(int(DADAPY_GRID_RANGE_MAX), X.shape[0] - 1)
    if range_max < 2:
        return float("nan")
    d.compute_distances(maxk=range_max)
    ids, _, _ = d.return_id_scaling_gride(range_max=range_max)
    return float(ids[-1])

def _skdim_factory(name: str):
    """Return a factory that builds a fresh skdim estimator each call, or None."""
    if not HAS_SKDIM: return None
    mapping = {
        "mom": MOM, "tle": TLE, "corrint": CorrInt, "fishers": FisherS,
        "lpca": lPCA, "lpca99": lPCA,
        "mle": MLE, "danco": DANCo, "mind_ml": MiND_ML,
        "mada": MADA, "knn": KNN, "ess": ESS,
    }
    cls = mapping.get(name)
    if cls is None: return None

    def _builder():
        if name == "lpca":      # FO variant
            return cls(ver="FO")
        elif name == "lpca99":  # ratio (0.99) variant
            return cls(ver="ratio", alphaRatio=0.99)
        else:
            return cls()
    return _builder

def _skdim_once_builder(name: str) -> Callable[[np.ndarray], float] | None:
    build = _skdim_factory(name)
    if build is None: return None

    def _once(X: np.ndarray) -> float:
        est = build()
        est.fit(_jitter_unique(X))
        return float(getattr(est, "dimension_", np.nan))
    return _once

# =============================== DATA ===============================
def load_word_df(csv_path: str, exclude_pos: set[str] = EXCLUDE_POS):
    df = pd.read_csv(csv_path, usecols=["sentence_id","tokens","pos"])
    df["sentence_id"] = df["sentence_id"].astype(str)
    df.tokens = df.tokens.apply(_to_list); df.pos = df.pos.apply(_to_list)

    rows = []
    for sid, toks, poss in df[["sentence_id","tokens","pos"]].itertuples(index=False):
        for wid, (tok, p) in enumerate(zip(toks, poss)):
            if p not in exclude_pos:
                rows.append((sid, wid, p, tok))
    word_df = pd.DataFrame(rows, columns=["sentence_id","word_id","pos","word"])
    return df, word_df

def sample_raw(word_df: pd.DataFrame, per_pos_cap: int = RAW_MAX_PER_POS) -> pd.DataFrame:
    """Per-POS cap without frequency matching."""
    picks = []
    for p, sub in word_df.groupby("pos", sort=False):
        n = min(len(sub), per_pos_cap)
        picks.append(sub.sample(n, random_state=RAND_SEED, replace=False))
    return pd.concat(picks, ignore_index=True)




# =============================== EMBEDDING (GPT‑2) ===============================
def _num_hidden_layers(model) -> int:
    """Works for GPT‑2 (n_layer) and BERT-like (num_hidden_layers)."""
    n = getattr(model.config, "num_hidden_layers", None)
    if n is None:
        n = getattr(model.config, "n_layer", None)
    if n is None:
        raise ValueError("Cannot determine number of hidden layers from model.config")
    return int(n)

def _hidden_size(model) -> int:
    d = getattr(model.config, "hidden_size", None)
    if d is None:
        d = getattr(model.config, "n_embd", None)  # GPT‑2
    if d is None:
        raise ValueError("Cannot determine hidden size from model.config")
    return int(d)

def embed_subset(df_all_sentences: pd.DataFrame,
                 subset_df: pd.DataFrame,
                 baseline: str = BASELINE,
                 word_rep_mode: str = WORD_REP_MODE,
                 batch_size: int = BATCH_SIZE) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return reps (L,N,D) and filled mask (N,) for the selected tokens.
    Uses word_ids() to map subword tokens back to original word indices.
    For GPT‑2, we (1) set add_prefix_space=True and (2) map pad_token→eos_token.
    """
    df_all_sentences["sentence_id"] = df_all_sentences["sentence_id"].astype(str)
    subset_df["sentence_id"] = subset_df["sentence_id"].astype(str)

    # sid -> list[(global_idx, word_id)]
    by_sid: Dict[str, List[Tuple[int,int]]] = {}
    for gidx, (sid, wid) in enumerate(subset_df[["sentence_id","word_id"]].itertuples(index=False)):
        by_sid.setdefault(str(sid), []).append((gidx, int(wid)))

    # Materialize in batch order
    sids = list(by_sid.keys())
    df_sel = (df_all_sentences[df_all_sentences.sentence_id.isin(sids)]
              .drop_duplicates("sentence_id")
              .set_index("sentence_id")
              .loc[sids])

    # GPT‑2 fast tokenizer with prefix space (needed with is_split_into_words)
    tokzr = AutoTokenizer.from_pretrained(baseline, use_fast=True, add_prefix_space=True)

    # Ensure padding works: GPT‑2 has no pad token by default
    if tokzr.pad_token is None:
        tokzr.pad_token = tokzr.eos_token

    enc_kwargs = dict(is_split_into_words=True, return_tensors="pt", padding=True)
    # (Fast tokenizers support `word_ids`; we already set add_prefix_space=True above)

    # Model
    model = AutoModel.from_pretrained(baseline, output_hidden_states=True).eval().to(device)
    # Make sure model's pad id is set (decoder-only models will ignore attention on padding anyway)
    if getattr(model.config, "pad_token_id", None) is None and tokzr.pad_token_id is not None:
        model.config.pad_token_id = tokzr.pad_token_id
    if device == "cuda":
        model.half()

    L = _num_hidden_layers(model) + 1   # include embedding layer
    D = _hidden_size(model)
    N = len(subset_df)

    reps   = np.zeros((L, N, D), np.float32)   # keep float32 to avoid duplicate-row issues in KNN metrics
    filled = np.zeros(N, dtype=bool)

    with torch.no_grad(), torch.cuda.amp.autocast(device == "cuda"):
        for start in tqdm(range(0, len(sids), batch_size), desc=f"{baseline} (embed subset)"):
            batch_ids    = sids[start : start + batch_size]
            batch_tokens = df_sel.loc[batch_ids, "tokens"].tolist()

            enc_be = tokzr(batch_tokens, **enc_kwargs)            # fast tokenizer
            enc_t  = {k: v.to(device) for k, v in enc_be.items()}
            out = model(**enc_t)
            # hidden_states: tuple len=L of (B,T,D)
            h = torch.stack(out.hidden_states).detach().cpu().numpy().astype(np.float32)  # (L,B,T,D)

            for b, sid in enumerate(batch_ids):
                # word_id -> token positions for this item
                mp: Dict[int, List[int]] = {}
                # word_ids(b) works only with fast tokenizers
                for tidx, wid in enumerate(enc_be.word_ids(b)):
                    if wid is not None:
                        mp.setdefault(int(wid), []).append(int(tidx))

                for gidx, wid in by_sid.get(sid, []):
                    toks = mp.get(wid)
                    if not toks:
                        continue
                    if word_rep_mode == "last":
                        vec = h[:, b, toks[-1], :]          # last subtoken
                    elif word_rep_mode == "mean":
                        vec = h[:, b, toks, :].mean(axis=1) # mean over subtokens
                    else:  # fallback to last
                        vec = h[:, b, toks[-1], :]
                    reps[:, gidx, :] = vec.astype(np.float32, copy=False)
                    filled[gidx] = True

            # free batch buffers
            del enc_be, enc_t, out, h
            if device == "cuda":
                torch.cuda.empty_cache()

    missing = int((~filled).sum())
    if missing:
        print(f"⚠ Missing vectors for {missing} of {N} sampled words")
    del model; gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    return reps, filled

# =============================== BOOTSTRAP CORE ===============================
def _bs_layer_loop(rep_sub: np.ndarray, M: int, n_reps: int, compute_once: Callable[[np.ndarray], float]):
    """Bootstrap: sample M with replacement and apply compute_once(X_layer) -> scalar for each layer."""
    L, N, D = rep_sub.shape
    rng = np.random.default_rng(RAND_SEED)
    A = np.full((n_reps, L), np.nan, np.float32)
    for r in range(n_reps):
        idx = rng.integers(0, N, size=M)
        for l in range(L):
            X = rep_sub[l, idx].astype(np.float32, copy=False)
            try:
                A[r, l] = float(compute_once(X))
            except Exception:
                A[r, l] = np.nan
    mu = np.nanmean(A, axis=0).astype(np.float32)
    lo = np.nanpercentile(A, 2.5, axis=0).astype(np.float32)
    hi = np.nanpercentile(A, 97.5, axis=0).astype(np.float32)
    return mu, lo, hi

# fast metric registry
FAST_ONCE: Dict[str, Callable[[np.ndarray], float]] = {
    "iso": _iso_once,
    "spect": _spect_once,
    "rand": _rand_once,
    "sf": _sf_once,
    "pfI": _pfI_once,
    "vmf_kappa": _vmf_kappa_once,
    "erank": _erank_once,
    "pr": _pr_once,
    "stable_rank": _stable_rank_once,
    "pca95": _pca95_once,
    "pca99": _pca99_once,
}

# heavy metric registry
HEAVY_ONCE: Dict[str, Callable[[np.ndarray], float] | None] = {
    "twonn": _dadapy_twonn_once,
    "gride": _dadapy_gride_once,
    "mom":   _skdim_once_builder("mom"),
    "tle":   _skdim_once_builder("tle"),
    "corrint": _skdim_once_builder("corrint"),
    "fishers": _skdim_once_builder("fishers"),
    "lpca":  _skdim_once_builder("lpca"),
    "lpca95": _skdim_once_builder("lpca99"),
    "lpca99": _skdim_once_builder("lpca99"),
    "mle":   _skdim_once_builder("mle"),
    "danco": _skdim_once_builder("danco"),
    "ess":   _skdim_once_builder("ess"),
    "mada":  _skdim_once_builder("mada"),
    "knn":   _skdim_once_builder("knn"),
}

LABELS = {
    # Isotropy
    "iso":"IsoScore","sf":"Spectral Flatness","pfI":"Partition Isotropy I",
    "vmf_kappa":"vMF κ","spect":"Spectral Ratio","rand":"RandCos |μ|",
    # Linear ID
    "erank":"Effective Rank","pr":"Participation Ratio","stable_rank":"Stable Rank",
    # Non-linear
    "twonn":"TwoNN ID","gride":"GRIDE",
    "mom":"MOM","tle":"TLE","corrint":"CorrInt","fishers":"FisherS",
    "lpca":"lPCA FO","lpca99":"lPCA 0.99","lpca95":"lPCA 0.95", "mle":"MLE","danco":"DANCo",
    "ess":"ESS","mada":"MADA","knn":"KNN",
}

# Choose plotting order
PLOT_ORDER = (
    "iso","sf","pfI","vmf_kappa","spect","rand",
    "erank","pr","stable_rank","pca95","pca99",
    "twonn","gride","mom","tle","corrint","fishers","lpca","lpca99",
    "mle","danco","ess","mada","knn"
)
#ALL_METRICS = list(PLOT_ORDER)
ALL_METRICS =["gride"]

# =============================== SAVE / PLOT ===============================
def save_metric_csv_all_pos(metric: str,
                            pos_to_stats: Dict[str, Dict[str, np.ndarray]],
                            layers: np.ndarray,
                            baseline: str,
                            subset_name: str = "raw"):
    rows = []
    for p, stats in pos_to_stats.items():
        mu, lo, hi, n = stats["mean"], stats.get("lo"), stats.get("hi"), stats.get("n", np.nan)
        for l, val in enumerate(mu):
            rows.append({
                "subset": subset_name, "model": baseline, "feature": "pos",
                "class": p, "metric": metric, "layer": int(layers[l]),
                "mean": float(val) if np.isfinite(val) else np.nan,
                "ci_low": float(lo[l]) if isinstance(lo, np.ndarray) and np.isfinite(lo[l]) else np.nan,
                "ci_high": float(hi[l]) if isinstance(hi, np.ndarray) and np.isfinite(hi[l]) else np.nan,
                "n_tokens": int(stats.get("n", 0)), "word_rep_mode": WORD_REP_MODE,
                "source_csv": Path(CSV_PATH).name,
            })
    df = pd.DataFrame(rows)
    out = CSV_DIR / f"pos_{subset_name}_{metric}_{baseline}.csv"
    df.to_csv(out, index=False)

def plot_metric_with_ci(pos_to_stats: Dict[str, Dict[str, np.ndarray]],
                        layers: np.ndarray, metric: str, title: str, out_path: Path):
    plt.figure(figsize=(9, 5))
    for p, stats in pos_to_stats.items():
        mu, lo, hi = stats["mean"], stats.get("lo"), stats.get("hi")
        if mu is None or np.all(np.isnan(mu)): continue
        plt.plot(layers, mu, label=p, lw=1.8)
        if isinstance(lo, np.ndarray) and isinstance(hi, np.ndarray) and not np.all(np.isnan(lo)):
            plt.fill_between(layers, lo, hi, alpha=0.15)
    plt.xlabel("Layer"); plt.ylabel(LABELS.get(metric, metric.upper())); plt.title(title)
    plt.legend(ncol=2, fontsize="small", title="POS", frameon=False)
    plt.tight_layout(); plt.savefig(out_path, dpi=220); plt.close()

# =============================== DRIVER ===============================
def run_pos_pipeline():
    # 1) Load
    df_all, word_df = load_word_df(CSV_PATH, EXCLUDE_POS)
    POS_TAGS = sorted(word_df.pos.unique())
    print(f"✓ corpus ready — {len(word_df):,} tokens across {len(POS_TAGS)} POS")
    print(f"• DADApy: {'available' if HAS_DADAPY else 'missing'}  • scikit-dimension: {'available' if HAS_SKDIM else 'missing'}")

    # 2) Raw sampling only (no frequency match)
    raw_df = sample_raw(word_df, RAW_MAX_PER_POS)
    print("Sample sizes per POS (raw cap):")
    print(raw_df.pos.value_counts().to_dict())

    # 3) Embed once (GPT‑2, last/mean subtoken per word)
    reps, filled = embed_subset(df_all, raw_df, BASELINE, WORD_REP_MODE, BATCH_SIZE)
    raw_df = raw_df.reset_index(drop=True).loc[filled].reset_index(drop=True)
    pos_arr = raw_df.pos.values
    L = reps.shape[0]; layers = np.arange(L)
    print(f"✓ embedded {len(raw_df):,} tokens  • layers={L}")

    # 4) Metric-by-metric loop (incremental outputs)
    for metric in ALL_METRICS:
        print(f"\n→ Computing metric: {metric} …")

        # Decide registry & bootstrap settings
        if metric in FAST_ONCE:
            compute_once = FAST_ONCE[metric]
            n_bs = N_BOOTSTRAP_FAST
            Mcap = FAST_BS_MAX_SAMP_PER_POS
        else:
            compute_once = HEAVY_ONCE.get(metric)
            n_bs = N_BOOTSTRAP_HEAVY
            Mcap = HEAVY_BS_MAX_SAMP_PER_POS

        if compute_once is None:
            print(f"  (skipping {metric}: estimator unavailable)")
            continue

        # Per-POS bootstrap
        metric_results: Dict[str, Dict[str, np.ndarray]] = {}
        for p in POS_TAGS:
            idx = np.where(pos_arr == p)[0]
            if idx.size < 3:
                continue
            sub = reps[:, idx]  # (L, n_p, D)
            Np = sub.shape[1]
            M = min(Mcap, Np)

            mu, lo, hi = _bs_layer_loop(sub, M, n_bs, compute_once)
            metric_results[p] = {"mean": mu, "lo": lo, "hi": hi, "n": int(Np)}

        # Save + plot immediately for this metric
        save_metric_csv_all_pos(metric, metric_results, layers, BASELINE, subset_name="raw")
        plot_metric_with_ci(metric_results, layers, metric,
                    title=f"{LABELS.get(metric, metric.upper())} • {BASELINE}",
                    out_path=PLOT_DIR / f"raw_{metric}_{BASELINE}.png")
        print(f"  ✓ saved: CSV= tables/pos_bootstrap/pos_raw_{metric}_{BASELINE}.csv  "
              f"plot= AO_POS/raw_{metric}_{BASELINE}_{WORD_REP_MODE}.png")

        # light cleanup
        del metric_results; gc.collect()
        if device == "cuda": torch.cuda.empty_cache()

    # Cleanup
    del reps; gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    print("\n✓ done (incremental outputs produced per metric).")

if __name__ == "__main__":
    run_pos_pipeline()


## No index

In [ ]:
import os, gc, ast, random, inspect
from pathlib import Path
from typing import Dict, List, Tuple, Callable

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel, GPT2TokenizerFast

# ============== Optional deps (gracefully skipped if not installed) ==============
HAS_DADAPY = False
try:
    from dadapy import Data  # DADApy ID estimators (TwoNN, GRIDE)
    HAS_DADAPY = True
except Exception:
    pass

HAS_SKDIM = False
try:
    from skdim.id import (
        MOM, TLE, CorrInt, FisherS, lPCA,
        MLE, DANCo, ESS, MiND_ML, MADA, KNN
    )
    HAS_SKDIM = True
except Exception:
    pass

# IsoScore: use library if available, else a simple monotone fallback
try:
    from IsoScore import IsoScore
    _HAS_ISOSCORE = True
except Exception:
    _HAS_ISOSCORE = False
    class _IsoScoreFallback:
        @staticmethod
        def IsoScore(X: np.ndarray) -> float:
            C = np.cov(X.T, ddof=0)
            ev = np.linalg.eigvalsh(C)
            if ev.mean() <= 0 or ev[-1] <= 0:
                return 0.0
            # mean/peak eigenvalue ratio in [0,1]; higher ≈ more isotropic
            return float(np.clip(ev.mean() / (ev[-1] + 1e-12), 0.0, 1.0))
    IsoScore = _IsoScoreFallback()

# =============================== CONFIG ===============================
CSV_PATH        = _resolve_csv_path("en_ewt-ud-train_sentences.csv")
BASELINE        = "gpt2"                 # ← GPT‑2
WORD_REP_MODE   = "last"                 # ← {"last","mean"} word representation
EXCLUDE_POS     = {"X", "SYM", "PART", "INTJ"}

# NEW: drop tokens at 0-based sentence index == 1 (the 2nd token)
# If you meant first token (1‑based “1”), change the check wid == 1 → wid == 0 in load_word_df.
EXCLUDE_INDEX_1 = True

# Sampling cap per POS for plotting (increase if you want)
RAW_MAX_PER_POS = int(1e12)         # effectively no cap


# Bootstrap replicates
N_BOOTSTRAP_FAST   = 50
N_BOOTSTRAP_HEAVY  = 100

# Per-replicate sample size M (min(cap, N_pos))
FAST_BS_MAX_SAMP_PER_POS  = int(1e12)
HEAVY_BS_MAX_SAMP_PER_POS = 5000

DADAPY_GRID_RANGE_MAX     = 64

# Runtime / output
BATCH_SIZE   = 8
RAND_SEED    = 42
PLOT_DIR     = _project_out("AO_POS"); PLOT_DIR.mkdir(exist_ok=True, parents=True)
CSV_DIR      = _project_out("tables") / "pos_bootstrap"; CSV_DIR.mkdir(exist_ok=True, parents=True)

# Isotropy extras
PFI_DIRS = 256
PFI_Q_LO = 5.0
PFI_Q_HI = 95.0
EPS      = 1e-12

# Repro & device
os.environ["TOKENIZERS_PARALLELISM"] = "true"
random.seed(RAND_SEED); np.random.seed(RAND_SEED); torch.manual_seed(RAND_SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda": torch.backends.cudnn.benchmark = True

# Seaborn style
sns.set_style("darkgrid")
plt.rcParams["figure.dpi"] = 120

# =============================== HELPERS ===============================
def _to_list(x):
    return ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x

def _center(X: np.ndarray) -> np.ndarray:
    return X - X.mean(0, keepdims=True)

def _eigvals_from_X(X: np.ndarray) -> np.ndarray:
    """Eigenvalues of covariance up to a constant via SVD of centered X (descending)."""
    Xc = _center(X.astype(np.float32, copy=False))
    try:
        _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        lam = (S**2).astype(np.float64)
        lam.sort()
        return lam[::-1]
    except Exception:
        return np.array([], dtype=np.float64)

def _jitter_unique(X: np.ndarray, eps: float = 1e-7) -> np.ndarray:
    """Add tiny noise if there are duplicate rows (helps NN-based estimators)."""
    try:
        if np.unique(X, axis=0).shape[0] < X.shape[0]:
            X = X + np.random.normal(scale=eps, size=X.shape).astype(X.dtype)
    except Exception:
        pass
    return X

# ========= Single-shot compute functions used inside bootstrap =========
# --- Isotropy (fast) ---

def _fast_isoscore(X: np.ndarray) -> float:
    """Fast IsoScore from covariance eigenvalues; matches the package algorithm."""
    try:
        lam = _eigvals_from_X(X)
    except Exception:
        Xc = np.asarray(X, dtype=np.float32)
        if Xc.ndim != 2 or Xc.size == 0:
            return float("nan")
        Xc = Xc - Xc.mean(0, keepdims=True)
        try:
            _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        except Exception:
            return float("nan")
        lam = (S.astype(np.float64) ** 2)
        lam = np.sort(np.maximum(lam, 0.0))[::-1]

    lam = np.asarray(lam, dtype=np.float64)
    lam = lam[np.isfinite(lam) & (lam >= 0)]
    n = int(lam.size)
    if n < 2:
        return float("nan")
    norm = float(np.linalg.norm(lam))
    if norm <= 0:
        return 0.0

    root_n = np.sqrt(n)
    denom = np.sqrt(2.0 * (n - root_n))
    if denom <= 0:
        return float("nan")

    delta = float(np.linalg.norm((root_n * lam) / norm - 1.0) / denom)
    delta = float(np.clip(delta, 0.0, 1.0))
    phi = (n - (delta ** 2) * (n - root_n)) ** 2 / (n ** 2)
    iso = (n * phi - 1.0) / (n - 1.0)
    return float(np.clip(iso, 0.0, 1.0))


def _iso_once(X: np.ndarray) -> float:
    return float(_fast_isoscore(X))

def _spect_once(X: np.ndarray) -> float:
    ev = np.linalg.eigvalsh(np.cov(X.T, ddof=0))
    return float(ev[-1] / (ev.mean() + 1e-9))

def _rand_once(X: np.ndarray, K: int = 2000) -> float:
    n = X.shape[0]
    if n < 2: return np.nan
    rng = np.random.default_rng()
    K_eff = min(K, (n*(n-1))//2)
    i = rng.integers(0, n, size=K_eff)
    j = rng.integers(0, n, size=K_eff)
    same = i == j
    if same.any():
        j[same] = rng.integers(0, n, size=same.sum())
    A, B = X[i], X[j]
    num = np.sum(A*B, axis=1)
    den = (np.linalg.norm(A, axis=1)*np.linalg.norm(B, axis=1) + 1e-9)
    return float(np.mean(np.abs(num/den)))

def _sf_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    gm = np.exp(np.mean(np.log(lam + EPS)))
    am = float(lam.mean() + EPS)
    return float(gm / am)

def _pfI_once(X: np.ndarray) -> float:
    n, d = X.shape
    if n < 2: return np.nan
    rng = np.random.default_rng()
    U = rng.standard_normal((PFI_DIRS, d)).astype(np.float32)
    U /= np.linalg.norm(U, axis=1, keepdims=True) + 1e-9
    S = U @ X.T
    m = np.max(S, axis=1, keepdims=True)
    logZ = (m + np.log(np.sum(np.exp(S - m), axis=1, keepdims=True))).ravel()
    lo = np.percentile(logZ, PFI_Q_LO)
    hi = np.percentile(logZ, PFI_Q_HI)
    return float(np.exp(lo - hi))  # ≈ min Z / max Z (robust)

def _vmf_kappa_once(X: np.ndarray) -> float:
    if X.shape[0] < 2: return np.nan
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    R = np.linalg.norm(Xn.mean(axis=0))
    d = Xn.shape[1]
    if R < 1e-9: return 0.0
    return float(max(R * (d - R**2) / (1.0 - R**2 + 1e-9), 0.0))

# --- Linear ID (fast) ---
def _pcaXX_once(X: np.ndarray, var_ratio: float) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    c = np.cumsum(lam); thr = c[-1] * var_ratio
    return float(np.searchsorted(c, thr) + 1)

def _pca95_once(X: np.ndarray) -> float:
    return _pcaXX_once(X, 0.95)

def _pca99_once(X: np.ndarray) -> float:
    return _pcaXX_once(X, 0.99)

def _erank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    p = lam / (lam.sum() + EPS)
    H = -(p * np.log(p + EPS)).sum()
    return float(np.exp(H))

def _pr_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    s1 = lam.sum(); s2 = (lam**2).sum()
    return float((s1**2) / (s2 + EPS))

def _stable_rank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    return float(lam.sum() / (lam.max() + EPS))

# --- Non-linear (heavy) ---
def _dadapy_twonn_once(X: np.ndarray) -> float:
    if not HAS_DADAPY: return np.nan
    d = Data(coordinates=_jitter_unique(X))
    id_est, _, _ = d.compute_id_2NN()
    return float(id_est)

def _dadapy_gride_once(X: np.ndarray) -> float:
    if not HAS_DADAPY: return np.nan
    d = Data(coordinates=_jitter_unique(X))
    range_max = min(int(DADAPY_GRID_RANGE_MAX), X.shape[0] - 1)
    if range_max < 2:
        return float("nan")
    d.compute_distances(maxk=range_max)
    ids, _, _ = d.return_id_scaling_gride(range_max=range_max)
    return float(ids[-1])

def _skdim_factory(name: str):
    """Return a factory that builds a fresh skdim estimator each call, or None."""
    if not HAS_SKDIM: return None
    mapping = {
        "mom": MOM, "tle": TLE, "corrint": CorrInt, "fishers": FisherS,
        "lpca": lPCA, "lpca99": lPCA,
        "mle": MLE, "danco": DANCo, "mind_ml": MiND_ML,
        "mada": MADA, "knn": KNN, "ess": ESS,
    }
    cls = mapping.get(name)
    if cls is None: return None

    def _builder():
        if name == "lpca":      # FO variant
            return cls(ver="FO")
        elif name == "lpca99":  # ratio (0.99) variant
            return cls(ver="ratio", alphaRatio=0.99)
        else:
            return cls()
    return _builder

def _skdim_once_builder(name: str) -> Callable[[np.ndarray], float] | None:
    build = _skdim_factory(name)
    if build is None: return None
    def _once(X: np.ndarray) -> float:
        est = build()
        est.fit(_jitter_unique(X))
        return float(getattr(est, "dimension_", np.nan))
    return _once

# =============================== DATA ===============================
def load_word_df(csv_path: str,
                 exclude_pos: set[str] = EXCLUDE_POS,
                 exclude_index_1: bool = EXCLUDE_INDEX_1):
    """
    Load sentence rows (tokens, pos), expand to token rows, filter by POS
    """
    df = pd.read_csv(csv_path, usecols=["sentence_id","tokens","pos"])
    df["sentence_id"] = df["sentence_id"].astype(str)
    df.tokens = df.tokens.apply(_to_list); df.pos = df.pos.apply(_to_list)

    rows = []
    for sid, toks, poss in df[["sentence_id","tokens","pos"]].itertuples(index=False):
        L = min(len(toks), len(poss))
        for wid in range(L):
            if exclude_index_1 and wid == 0:
                continue
            p = poss[wid]
            if p in exclude_pos:
                continue
            rows.append((sid, wid, p, toks[wid]))

    word_df = pd.DataFrame(rows, columns=["sentence_id","word_id","pos","word"])

    # Extra safety: enforce the filter even if logic above changes later
    if exclude_index_1 and not word_df.empty:
        word_df = word_df[word_df.word_id !=0].reset_index(drop=True)

    return df, word_df

def sample_raw(word_df: pd.DataFrame, per_pos_cap: int = RAW_MAX_PER_POS) -> pd.DataFrame:
    """Per-POS cap without frequency matching."""
    picks = []
    for p, sub in word_df.groupby("pos", sort=False):
        n = min(len(sub), per_pos_cap)
        picks.append(sub.sample(n, random_state=RAND_SEED, replace=False))
    return pd.concat(picks, ignore_index=True)

# =============================== MODEL LOADING (robust for GPT‑2) ===============================
def _load_tok_and_model(baseline: str):
    """
    Robust loader:
    - GPT‑2: prefer GPT2TokenizerFast; ensure right padding; set pad_token=eos_token if missing.
    - Try both 'gpt2' and 'openai-community/gpt2' (some envs only have one).
    - Set model.config.pad_token_id if missing.
    """
    candidates = [baseline]
    b = baseline.lower()
    if "gpt2" in b:
        if baseline != "openai-community/gpt2":
            candidates.append("openai-community/gpt2")
        if baseline != "gpt2":
            candidates.append("gpt2")

    last_err = None
    for mid in candidates:
        try:
            if "gpt2" in mid.lower():
                tok = GPT2TokenizerFast.from_pretrained(mid, add_prefix_space=True)
            else:
                tok = AutoTokenizer.from_pretrained(mid, use_fast=True, add_prefix_space=True)

            if getattr(tok, "padding_side", None) != "right":
                tok.padding_side = "right"
            if tok.pad_token is None and getattr(tok, "eos_token", None) is not None:
                tok.pad_token = tok.eos_token

            mdl = AutoModel.from_pretrained(mid, output_hidden_states=True)
            if getattr(mdl.config, "pad_token_id", None) is None and tok.pad_token_id is not None:
                mdl.config.pad_token_id = tok.pad_token_id

            mdl = mdl.eval().to(device)
            if device == "cuda":
                mdl.half()
            return tok, mdl, mid
        except Exception as e:
            last_err = e
            continue

    raise RuntimeError(f"Failed to load tokenizer/model for {candidates}. Last error: {last_err}")

# =============================== EMBEDDING ===============================
def _num_hidden_layers(model) -> int:
    n = getattr(model.config, "num_hidden_layers", None)
    if n is None: n = getattr(model.config, "n_layer", None)
    if n is None: raise ValueError("Cannot determine number of hidden layers from model.config")
    return int(n)

def _hidden_size(model) -> int:
    d = getattr(model.config, "hidden_size", None)
    if d is None: d = getattr(model.config, "n_embd", None)  # GPT‑2
    if d is None: raise ValueError("Cannot determine hidden size from model.config")
    return int(d)

def embed_subset(df_all_sentences: pd.DataFrame,
                 subset_df: pd.DataFrame,
                 baseline: str = BASELINE,
                 word_rep_mode: str = WORD_REP_MODE,
                 batch_size: int = BATCH_SIZE) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return reps (L,N,D) and filled mask (N,) for the selected tokens.
    Uses word_ids() to map subword tokens back to original word indices.
    """
    df_all_sentences["sentence_id"] = df_all_sentences["sentence_id"].astype(str)
    subset_df["sentence_id"] = subset_df["sentence_id"].astype(str)

    # sid -> list[(global_idx, word_id)]
    by_sid: Dict[str, List[Tuple[int,int]]] = {}
    for gidx, (sid, wid) in enumerate(subset_df[["sentence_id","word_id"]].itertuples(index=False)):
        by_sid.setdefault(str(sid), []).append((gidx, int(wid)))

    # Materialize in batch order
    sids = list(by_sid.keys())
    df_sel = (df_all_sentences[df_all_sentences.sentence_id.isin(sids)]
              .drop_duplicates("sentence_id")
              .set_index("sentence_id")
              .loc[sids])

    tokzr, model, model_id = _load_tok_and_model(baseline)

    enc_kwargs = dict(is_split_into_words=True, return_tensors="pt", padding=True)
    if "add_prefix_space" in inspect.signature(tokzr.__call__).parameters:
        enc_kwargs["add_prefix_space"] = True

    L = _num_hidden_layers(model) + 1   # include embedding layer
    D = _hidden_size(model)
    N = len(subset_df)

    reps   = np.zeros((L, N, D), np.float32)  # use float32 (stable for some estimators)
    filled = np.zeros(N, dtype=bool)

    with torch.no_grad(), torch.cuda.amp.autocast(device == "cuda"):
        for start in tqdm(range(0, len(sids), batch_size), desc=f"{model_id} (embed subset)"):
            batch_ids    = sids[start : start + batch_size]
            batch_tokens = df_sel.loc[batch_ids, "tokens"].tolist()

            enc_be = tokzr(batch_tokens, **enc_kwargs)            # fast tokenizer
            enc_t  = {k: v.to(device) for k, v in enc_be.items()}
            out = model(**enc_t)
            # hidden_states: tuple len=L of (B,T,D)
            h = torch.stack(out.hidden_states).detach().cpu().numpy().astype(np.float32)  # (L,B,T,D)

            for b, sid in enumerate(batch_ids):
                # word_id -> token positions for this item
                mp: Dict[int, List[int]] = {}
                wids = enc_be.word_ids(b)
                if wids is None:
                    raise RuntimeError("Fast tokenizer required; word_ids() returned None.")
                for tidx, wid in enumerate(wids):
                    if wid is not None:
                        mp.setdefault(int(wid), []).append(int(tidx))

                for gidx, wid in by_sid.get(sid, []):
                    toks = mp.get(wid)
                    if not toks:
                        continue
                    if word_rep_mode == "last":
                        vec = h[:, b, toks[-1], :]          # last subtoken
                    elif word_rep_mode == "mean":
                        vec = h[:, b, toks, :].mean(axis=1) # mean over subtokens
                    else:  # fallback to last for decoder models
                        vec = h[:, b, toks[-1], :]
                    reps[:, gidx, :] = vec.astype(np.float32, copy=False)
                    filled[gidx] = True

            del enc_be, enc_t, out, h
            if device == "cuda":
                torch.cuda.empty_cache()

    missing = int((~filled).sum())
    if missing:
        print(f"⚠ Missing vectors for {missing} of {N} sampled words")
    del model; gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    return reps, filled

# =============================== BOOTSTRAP CORE ===============================
def _bs_layer_loop(rep_sub: np.ndarray, M: int, n_reps: int, compute_once: Callable[[np.ndarray], float]):
    """Bootstrap: sample M with replacement and apply compute_once(X_layer) -> scalar for each layer."""
    L, N, D = rep_sub.shape
    rng = np.random.default_rng(RAND_SEED)
    A = np.full((n_reps, L), np.nan, np.float32)
    for r in range(n_reps):
        idx = rng.integers(0, N, size=M)
        for l in range(L):
            X = rep_sub[l, idx].astype(np.float32, copy=False)
            try:
                A[r, l] = float(compute_once(X))
            except Exception:
                A[r, l] = np.nan
    mu = np.nanmean(A, axis=0).astype(np.float32)
    lo = np.nanpercentile(A, 2.5, axis=0).astype(np.float32)
    hi = np.nanpercentile(A, 97.5, axis=0).astype(np.float32)
    return mu, lo, hi

# fast metric registry
FAST_ONCE: Dict[str, Callable[[np.ndarray], float]] = {
    "iso": _iso_once,
    "spect": _spect_once,
    "rand": _rand_once,
    "sf": _sf_once,
    "pfI": _pfI_once,
    "vmf_kappa": _vmf_kappa_once,
    "erank": _erank_once,
    "pr": _pr_once,
    "stable_rank": _stable_rank_once,
    "pca95": _pca95_once,
    "pca99": _pca99_once,
}

# heavy metric registry
HEAVY_ONCE: Dict[str, Callable[[np.ndarray], float] | None] = {
    "twonn": _dadapy_twonn_once,
    "gride": _dadapy_gride_once,
    "mom":   _skdim_once_builder("mom"),
    "tle":   _skdim_once_builder("tle"),
    "corrint": _skdim_once_builder("corrint"),
    "fishers": _skdim_once_builder("fishers"),
    "lpca":  _skdim_once_builder("lpca"),
    "lpca95": _skdim_once_builder("lpca99"),
    "lpca99": _skdim_once_builder("lpca99"),
    "mle":   _skdim_once_builder("mle"),
    "danco": _skdim_once_builder("danco"),
    "ess":   _skdim_once_builder("ess"),
    "mada":  _skdim_once_builder("mada"),
    "knn":   _skdim_once_builder("knn"),
}

LABELS = {
    # Isotropy
    "iso":"IsoScore","sf":"Spectral Flatness","pfI":"Partition Isotropy I",
    "vmf_kappa":"vMF κ","spect":"Spectral Ratio","rand":"RandCos |μ|",
    # Linear ID
    "erank":"Effective Rank","pr":"Participation Ratio","stable_rank":"Stable Rank",
    # Non-linear
    "twonn":"TwoNN ID","gride":"GRIDE",
    "mom":"MOM","tle":"TLE","corrint":"CorrInt","fishers":"FisherS",
    "lpca":"lPCA FO","lpca99":"lPCA 0.99","lpca95":"lPCA 0.95", "mle":"MLE","danco":"DANCo",
    "ess":"ESS","mada":"MADA","knn":"KNN",
}

# Choose plotting order
PLOT_ORDER = (
    "iso","sf","pfI","vmf_kappa","spect","rand",
    "erank","pr","stable_rank","pca95","pca99",
    "twonn","gride","mom","tle","corrint","fishers","lpca","lpca99",
    "mle","danco","ess","mada","knn"
)
#ALL_METRICS = list(PLOT_ORDER)
ALL_METRICS = ["iso"]

# =============================== SAVE / PLOT ===============================
def save_metric_csv_all_pos(metric: str,
                            pos_to_stats: Dict[str, Dict[str, np.ndarray]],
                            layers: np.ndarray,
                            baseline: str,
                            subset_name: str = "raw"):
    rows = []
    for p, stats in pos_to_stats.items():
        mu, lo, hi, n = stats["mean"], stats.get("lo"), stats.get("hi"), stats.get("n", np.nan)
        for l, val in enumerate(mu):
            rows.append({
                "subset": subset_name, "model": baseline, "feature": "pos",
                "class": p, "metric": metric, "layer": int(layers[l]),
                "mean": float(val) if np.isfinite(val) else np.nan,
                "ci_low": float(lo[l]) if isinstance(lo, np.ndarray) and np.isfinite(lo[l]) else np.nan,
                "ci_high": float(hi[l]) if isinstance(hi, np.ndarray) and np.isfinite(hi[l]) else np.nan,
                "n_tokens": int(stats.get("n", 0)), "word_rep_mode": WORD_REP_MODE,
                "source_csv": Path(CSV_PATH).name,
            })
    df = pd.DataFrame(rows)
    out = CSV_DIR / f"pos_{subset_name}_{metric}_{baseline}.csv"
    df.to_csv(out, index=False)

def plot_metric_with_ci(pos_to_stats: Dict[str, Dict[str, np.ndarray]],
                        layers: np.ndarray, metric: str, title: str, out_path: Path):
    plt.figure(figsize=(9, 5))
    for p, stats in pos_to_stats.items():
        mu, lo, hi = stats["mean"], stats.get("lo"), stats.get("hi")
        if mu is None or np.all(np.isnan(mu)): continue
        plt.plot(layers, mu, label=p, lw=1.8)
        if isinstance(lo, np.ndarray) and isinstance(hi, np.ndarray) and not np.all(np.isnan(lo)):
            plt.fill_between(layers, lo, hi, alpha=0.15)
    plt.xlabel("Layer"); plt.ylabel(LABELS.get(metric, metric.upper())); plt.title(title)
    plt.legend(ncol=2, fontsize="small", title="POS", frameon=False)
    plt.tight_layout(); plt.savefig(out_path, dpi=220); plt.close()

# =============================== DRIVER ===============================
def run_pos_pipeline():
    # 1) Load
    df_all, word_df = load_word_df(CSV_PATH, EXCLUDE_POS, EXCLUDE_INDEX_1)
    POS_TAGS = sorted(word_df.pos.unique())
    print(f"✓ corpus ready — {len(word_df):,} tokens across {len(POS_TAGS)} POS")
    print(f"• DADApy: {'available' if HAS_DADAPY else 'missing'}  • scikit-dimension: {'available' if HAS_SKDIM else 'missing'}")

    # 2) Raw sampling only (no frequency match)
    raw_df = sample_raw(word_df, RAW_MAX_PER_POS)
    print("Sample sizes per POS (raw cap):")
    print(raw_df.pos.value_counts().to_dict())

    # 3) Embed once (GPT‑2, last/mean subtoken per word)
    reps, filled = embed_subset(df_all, raw_df, BASELINE, WORD_REP_MODE, BATCH_SIZE)
    raw_df = raw_df.reset_index(drop=True).loc[filled].reset_index(drop=True)
    pos_arr = raw_df.pos.values
    L = reps.shape[0]; layers = np.arange(L)
    print(f"✓ embedded {len(raw_df):,} tokens  • layers={L}")

    # 4) Metric-by-metric loop (incremental outputs)
    for metric in ALL_METRICS:
        print(f"\n→ Computing metric: {metric} …")

        # Decide registry & bootstrap settings
        if metric in FAST_ONCE:
            compute_once = FAST_ONCE[metric]
            n_bs = N_BOOTSTRAP_FAST
            Mcap = FAST_BS_MAX_SAMP_PER_POS
        else:
            compute_once = HEAVY_ONCE.get(metric)
            n_bs = N_BOOTSTRAP_HEAVY
            Mcap = HEAVY_BS_MAX_SAMP_PER_POS

        if compute_once is None:
            print(f"  (skipping {metric}: estimator unavailable)")
            continue

        # Per-POS bootstrap
        metric_results: Dict[str, Dict[str, np.ndarray]] = {}
        for p in POS_TAGS:
            idx = np.where(pos_arr == p)[0]
            if idx.size < 3:
                continue
            sub = reps[:, idx]  # (L, n_p, D)
            Np = sub.shape[1]
            M = min(Mcap, Np)

            mu, lo, hi = _bs_layer_loop(sub, M, n_bs, compute_once)
            metric_results[p] = {"mean": mu, "lo": lo, "hi": hi, "n": int(Np)}

        # Save + plot immediately for this metric
        save_metric_csv_all_pos(metric, metric_results, layers, BASELINE, subset_name="raw")
        plot_metric_with_ci(metric_results, layers, metric,
                    title=f"{LABELS.get(metric, metric.upper())} • {BASELINE}",
                    out_path=PLOT_DIR / f"raw_{metric}_{BASELINE}.png")
        print(f"  ✓ saved: CSV= tables/pos_bootstrap/pos_raw_{metric}_{BASELINE}.csv  "
              f"plot= AO_POS/raw_{metric}_{BASELINE}_{WORD_REP_MODE}.png")

        del metric_results; gc.collect()
        if device == "cuda": torch.cuda.empty_cache()

    # Cleanup
    del reps; gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    print("\n✓ done (incremental outputs produced per metric).")

if __name__ == "__main__":
    run_pos_pipeline()



# all points

In [ ]:

import os, gc, ast, random, inspect, contextlib
from pathlib import Path
from typing import Dict, List, Tuple, Callable, Optional

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModel

# ---------------- optional deps ----------------
HAS_DADAPY = False
try:
    from dadapy import Data
    HAS_DADAPY = True
except Exception:
    pass

HAS_SKDIM = False
try:
    from skdim.id import MOM, TLE, CorrInt, FisherS, lPCA, MLE, DANCo, ESS, MiND_ML, MADA, KNN
    HAS_SKDIM = True
except Exception:
    pass

# --- IsoScore (PyPI package name: IsoScore) ---
# Correct usage: from IsoScore.IsoScore import * ; IsoScore(points)  (see PyPI docs)
# We'll wrap it so the rest of the code can call iso_score(X).
_HAS_ISOSCORE = False
try:
    from IsoScore.IsoScore import IsoScore as _IsoScore_fn
    _HAS_ISOSCORE = True
except Exception:
    _IsoScore_fn = None
    _HAS_ISOSCORE = False


def _fast_isoscore(X: np.ndarray) -> float:
    """Fast IsoScore from covariance eigenvalues; matches the package algorithm."""
    try:
        lam = _eigvals_from_X(X)
    except Exception:
        Xc = np.asarray(X, dtype=np.float32)
        if Xc.ndim != 2 or Xc.size == 0:
            return float("nan")
        Xc = Xc - Xc.mean(0, keepdims=True)
        try:
            _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        except Exception:
            return float("nan")
        lam = (S.astype(np.float64) ** 2)
        lam = np.sort(np.maximum(lam, 0.0))[::-1]

    lam = np.asarray(lam, dtype=np.float64)
    lam = lam[np.isfinite(lam) & (lam >= 0)]
    n = int(lam.size)
    if n < 2:
        return float("nan")
    norm = float(np.linalg.norm(lam))
    if norm <= 0:
        return 0.0

    root_n = np.sqrt(n)
    denom = np.sqrt(2.0 * (n - root_n))
    if denom <= 0:
        return float("nan")

    delta = float(np.linalg.norm((root_n * lam) / norm - 1.0) / denom)
    delta = float(np.clip(delta, 0.0, 1.0))
    phi = (n - (delta ** 2) * (n - root_n)) ** 2 / (n ** 2)
    iso = (n * phi - 1.0) / (n - 1.0)
    return float(np.clip(iso, 0.0, 1.0))


def iso_score(X: np.ndarray) -> float:
    """
    Returns IsoScore in [0,1]. Uses IsoScore package if available; otherwise a stable fallback.
    """
    X = np.asarray(X, dtype=np.float32)
    if _HAS_ISOSCORE and _IsoScore_fn is not None:
        # The reference implementation accepts torch tensors; numpy usually works too,
        # but torch is safest across versions.
        return float(_fast_isoscore(X))
    # Fallback: eigen-spectrum mean/max ratio (monotone proxy, not identical to IsoScore)
    C = np.cov(X.T, ddof=0)
    ev = np.linalg.eigvalsh(C)
    if ev.size == 0 or ev.mean() <= 0 or ev[-1] <= 0:
        return 0.0
    return float(np.clip(ev.mean() / ev[-1], 0.0, 1.0))

# ---------------- style ----------------
sns.set_style("darkgrid")
plt.rcParams["figure.dpi"] = 120


In [ ]:
# =============================== CONFIG ===============================
CSV_PATH    = _resolve_csv_path("en_ewt-ud-train_sentences.csv")

from pathlib import Path

# IMPORTANT: avoid "gpt2" short id to bypass any local ./gpt2 folder shadowing
MODELS = ["bert-base-uncased", "openai-community/gpt2"]

# Quick check: do you have a local folder called "gpt2"?
print("Local ./gpt2 exists?", Path("gpt2").exists(), " (cwd:", Path(".").resolve(), ")")
if Path("gpt2").exists():
    print("⚠️ Rename or remove the local ./gpt2 folder (it can shadow the HF model id).")
REP_MODES   = ["first", "mean", "last"]
EXCLUDE_POS = {"X", "SYM", "PART", "INTJ"}

# If None, uses ALL tokens (can be heavy). Strongly recommend setting a cap for RAM safety.
ALL_TOKEN_CAP = None

BATCH_SIZE  = 8
RAND_SEED   = 42

# Bootstrap replicates (keep small while debugging)
N_BOOTSTRAP_FAST  = 10
N_BOOTSTRAP_HEAVY = 20

# Per bootstrap replicate sample size M
FAST_BS_MAX_SAMP  = 10_000
HEAVY_BS_MAX_SAMP = 5_000   # match your paper's heavy cap idea

# GRIDE max neighbor rank
DADAPY_GRID_RANGE_MAX = 64

# Isotropy helpers
PFI_DIRS = 128
PFI_Q_LO = 5.0
PFI_Q_HI = 95.0
EPS      = 1e-12

# Output
OUT_DIR  = _project_out("metrics_all_token"); OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = OUT_DIR / "plots";  PLOT_DIR.mkdir(exist_ok=True)
CSV_DIR  = OUT_DIR / "tables"; CSV_DIR.mkdir(exist_ok=True)

MODEL_LABEL = {
    "bert-base-uncased": "BERT-base",
    "gpt2": "GPT-2",
}

def _model_tag(name: str) -> str:
    return name.split("/")[-1].replace(":", "_")

# Repro & device
os.environ["TOKENIZERS_PARALLELISM"] = "true"
random.seed(RAND_SEED); np.random.seed(RAND_SEED); torch.manual_seed(RAND_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.backends.cudnn.benchmark = True


In [ ]:
def _to_list(x):
    return ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x

def _center(X: np.ndarray) -> np.ndarray:
    return X - X.mean(0, keepdims=True)

def _eigvals_from_X(X: np.ndarray) -> np.ndarray:
    """Eigenvalues of covariance up to a constant via SVD of centered X (descending)."""
    Xc = _center(np.asarray(X, dtype=np.float32))
    try:
        _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        lam = (S**2).astype(np.float64)
        lam.sort()
        return lam[::-1]
    except Exception:
        return np.array([], dtype=np.float64)

def _prep_for_knn(
    X: np.ndarray,
    cap: Optional[int],
    rng: np.random.Generator,
    jitter_scale: float = 1e-6,
) -> np.ndarray:
    """
    For NN-based ID estimators:
      1) float32 contiguous
      2) optional cap (subsample without replacement)
      3) dedup exact rows (helps DADApy)
      4) add tiny jitter to break any remaining ties / bootstrap duplicates
    """
    X = np.ascontiguousarray(X, dtype=np.float32)

    if cap is not None and X.shape[0] > cap:
        idx = rng.choice(X.shape[0], size=cap, replace=False)
        X = X[idx]

    # Deduplicate exact identical points
    try:
        X = np.unique(X, axis=0)
    except Exception:
        pass

    # Add jitter (scaled to std) to break bootstrap duplicates / ties
    std = float(X.std()) or 1.0
    eps = jitter_scale * std
    X = X + rng.normal(0.0, eps, size=X.shape).astype(np.float32)

    return X


In [ ]:
# ========= Isotropy (fast) =========
def _iso_once(X: np.ndarray) -> float:
    return float(iso_score(X))

def _spect_once(X: np.ndarray) -> float:
    ev = np.linalg.eigvalsh(np.cov(np.asarray(X, dtype=np.float32).T, ddof=0))
    return float(ev[-1] / (ev.mean() + EPS))

def _rand_once(X: np.ndarray, K: int = 2000) -> float:
    X = np.asarray(X, dtype=np.float32)
    n = X.shape[0]
    if n < 2:
        return np.nan
    rng = np.random.default_rng(RAND_SEED)
    K_eff = min(K, (n * (n - 1)) // 2)
    i = rng.integers(0, n, size=K_eff)
    j = rng.integers(0, n, size=K_eff)
    same = (i == j)
    if same.any():
        j[same] = rng.integers(0, n, size=int(same.sum()))
    A, B = X[i], X[j]
    num = np.sum(A * B, axis=1)
    den = (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-9)
    return float(np.mean(np.abs(num / den)))

def _sf_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return np.nan
    gm = np.exp(np.mean(np.log(lam + EPS)))
    am = float(lam.mean() + EPS)
    return float(gm / am)

def _pfI_once(X: np.ndarray) -> float:
    X = np.asarray(X, dtype=np.float32)
    n, d = X.shape
    if n < 2:
        return np.nan
    rng = np.random.default_rng(RAND_SEED)
    U = rng.standard_normal((PFI_DIRS, d)).astype(np.float32)
    U /= np.linalg.norm(U, axis=1, keepdims=True) + 1e-9
    S = U @ X.T
    m = np.max(S, axis=1, keepdims=True)
    logZ = (m + np.log(np.sum(np.exp(S - m), axis=1, keepdims=True))).ravel()
    lo = np.percentile(logZ, PFI_Q_LO)
    hi = np.percentile(logZ, PFI_Q_HI)
    return float(np.exp(lo - hi))

def _vmf_kappa_once(X: np.ndarray) -> float:
    X = np.asarray(X, dtype=np.float32)
    if X.shape[0] < 2:
        return np.nan
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    R = np.linalg.norm(Xn.mean(axis=0))
    d = Xn.shape[1]
    if R < 1e-9:
        return 0.0
    return float(max(R * (d - R**2) / (1.0 - R**2 + 1e-9), 0.0))

# ========= Linear ID (fast) =========
def _erank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return np.nan
    p = lam / (lam.sum() + EPS)
    H = -(p * np.log(p + EPS)).sum()
    return float(np.exp(H))

def _pr_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return np.nan
    s1 = lam.sum()
    s2 = (lam**2).sum()
    return float((s1**2) / (s2 + EPS))

def _stable_rank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return np.nan
    return float(lam.sum() / (lam.max() + EPS))

# ========= Nonlinear ID (heavy) =========
def _dadapy_twonn_once(X: np.ndarray) -> float:
    if not HAS_DADAPY:
        return np.nan
    rng = np.random.default_rng(RAND_SEED)
    Xp = _prep_for_knn(X, cap=HEAVY_BS_MAX_SAMP, rng=rng)
    if Xp.shape[0] < 3:
        return np.nan
    d = Data(coordinates=Xp)
    # DADApy supports removing identical points explicitly
    try:
        d.remove_identical_points()
    except Exception:
        pass
    id_est, _, _ = d.compute_id_2NN()
    return float(id_est)

def _dadapy_gride_once(X: np.ndarray) -> float:
    if not HAS_DADAPY:
        return np.nan
    rng = np.random.default_rng(RAND_SEED)
    Xp = _prep_for_knn(X, cap=HEAVY_BS_MAX_SAMP, rng=rng)
    if Xp.shape[0] < 20:
        return np.nan
    d = Data(coordinates=Xp)
    try:
        d.remove_identical_points()
    except Exception:
        pass
    range_max = min(int(DADAPY_GRID_RANGE_MAX), X.shape[0] - 1)
    if range_max < 2:
        return float("nan")
    d.compute_distances(maxk=range_max)
    ids, _, _ = d.return_id_scaling_gride(range_max=range_max)
    return float(ids[-1])

def _skdim_factory(name: str):
    if not HAS_SKDIM:
        return None
    mapping = {
        "mom": MOM, "tle": TLE, "corrint": CorrInt, "fishers": FisherS,
        "lpca": lPCA, "lpca95": lPCA, "lpca99": lPCA,
        "mle": MLE, "danco": DANCo, "mind_ml": MiND_ML,
        "mada": MADA, "knn": KNN,
    }
    cls = mapping.get(name)
    if cls is None:
        return None

    def _builder():
        if name == "lpca":
            return cls(ver="FO")
        if name == "lpca95":
            return cls(ver="ratio", alphaRatio=0.95)
        if name == "lpca99":
            return cls(ver="ratio", alphaRatio=0.99)
        return cls()

    return _builder

def _skdim_once_builder(name: str):
    build = _skdim_factory(name)
    if build is None:
        return None

    def _once(X: np.ndarray) -> float:
        rng = np.random.default_rng(RAND_SEED)
        Xp = _prep_for_knn(X, cap=HEAVY_BS_MAX_SAMP, rng=rng)
        if Xp.shape[0] < 3:
            return np.nan
        est = build()
        est.fit(Xp)
        return float(getattr(est, "dimension_", np.nan))

    return _once

FAST_ONCE: Dict[str, Callable[[np.ndarray], float]] = {
    "iso": _iso_once,
    "sf": _sf_once,
    "pfi": _pfI_once,
    "vmf_kappa": _vmf_kappa_once,
    "spect": _spect_once,
    "rand": _rand_once,
    "erank": _erank_once,
    "pr": _pr_once,
    "stable_rank": _stable_rank_once,
}

HEAVY_ONCE: Dict[str, Optional[Callable[[np.ndarray], float]]] = {
    "twonn": _dadapy_twonn_once,
    "gride": _dadapy_gride_once,
    "mom": _skdim_once_builder("mom"),
    "tle": _skdim_once_builder("tle"),
    "corrint": _skdim_once_builder("corrint"),
    "fishers": _skdim_once_builder("fishers"),
    "lpca": _skdim_once_builder("lpca"),
    "lpca95": _skdim_once_builder("lpca95"),
    "lpca99": _skdim_once_builder("lpca99"),
    "mle": _skdim_once_builder("mle"),
    "mada": _skdim_once_builder("mada"),
}

LABELS = {
    # Isotropy
    "iso": "IsoScore",
    "sf": "Spectral Flatness",
    "pfi": "PFI",
    "vmf_kappa": "vMF κ",
    "spect": "Spectral Ratio",
    "rand": "RandCos |μ|",
    # Linear ID
    "erank": "Effective Rank",
    "pr": "Participation Ratio",
    "stable_rank": "Stable Rank",
    "lpca": "lPCA (FO)",
    "lpca95": "lPCA 0.95",
    "lpca99": "lPCA 0.99",
    # Non-linear
    "twonn": "TwoNN ID",
    "gride": "GRIDE",
    "mom": "MOM",
    "tle": "TLE",
    "corrint": "CorrInt",
    "fishers": "FisherS",
    "mle": "MLE",
    "mada": "MADA",
}

PLOT_ORDER = (
    "iso","sf","pfi","vmf_kappa","spect","rand",
    "erank","pr","stable_rank",
    "twonn","gride","mom","tle","corrint","fishers","lpca","lpca95","lpca99","mle","mada"
)

# Choose which metrics to run (start small; add as libs are available)
ALL_METRICS = ["iso","gride", "lpca99"]


In [ ]:
def load_word_df(csv_path: str, exclude_pos: set[str] = EXCLUDE_POS):
    df = pd.read_csv(csv_path, usecols=["sentence_id", "tokens", "pos"])
    df["sentence_id"] = df["sentence_id"].astype(str)
    df["tokens"] = df["tokens"].apply(_to_list)
    df["pos"] = df["pos"].apply(_to_list)

    rows = []
    for sid, toks, poss in df[["sentence_id", "tokens", "pos"]].itertuples(index=False):
        for wid, p in enumerate(poss):
            if (exclude_pos is None) or (p not in exclude_pos):
                rows.append((sid, wid))
    word_df = pd.DataFrame(rows, columns=["sentence_id", "word_id"])
    return df, word_df

def sample_all(word_df: pd.DataFrame, cap: Optional[int]) -> pd.DataFrame:
    if cap is None or len(word_df) <= cap:
        return word_df.reset_index(drop=True)
    return word_df.sample(cap, random_state=RAND_SEED).reset_index(drop=True)


In [ ]:
def _load_tokenizer(model_name: str):
    # GPT-2 fast tokenizer needs add_prefix_space=True for pretokenized inputs
    try:
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True, add_prefix_space=True)
    except TypeError:
        tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        # some tokenizers expose this attribute
        if hasattr(tok, "add_prefix_space"):
            try:
                tok.add_prefix_space = True
            except Exception:
                pass

    # GPT-2 has no PAD token -> use EOS
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def embed_subset(
    df_all_sentences: pd.DataFrame,
    subset_df: pd.DataFrame,
    baseline: str,
    word_rep_mode: str,
    batch_size: int = BATCH_SIZE,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return reps (L,N,D) float16 and filled mask for the selected tokens.
    """
    df_all_sentences = df_all_sentences.copy()
    df_all_sentences["sentence_id"] = df_all_sentences["sentence_id"].astype(str)

    subset_df = subset_df.copy()
    subset_df["sentence_id"] = subset_df["sentence_id"].astype(str)

    # sid -> list[(global_idx, word_id)]
    by_sid: Dict[str, List[Tuple[int, int]]] = {}
    for gidx, (sid, wid) in enumerate(subset_df[["sentence_id", "word_id"]].itertuples(index=False)):
        by_sid.setdefault(str(sid), []).append((gidx, int(wid)))

    sids = list(by_sid.keys())
    df_sel = (
        df_all_sentences[df_all_sentences.sentence_id.isin(sids)]
        .drop_duplicates("sentence_id")
        .set_index("sentence_id")
        .loc[sids]
    )

    tokzr = _load_tokenizer(baseline)

    enc_kwargs = dict(
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )

    model = AutoModel.from_pretrained(baseline, output_hidden_states=True).eval().to(device)

    # ensure pad_token_id exists for GPT2-like models
    if getattr(model.config, "pad_token_id", None) is None and tokzr.pad_token_id is not None:
        model.config.pad_token_id = tokzr.pad_token_id

    if device == "cuda":
        model.half()

    L = model.config.num_hidden_layers + 1
    D = model.config.hidden_size
    N = len(subset_df)

    reps = np.zeros((L, N, D), np.float16)
    filled = np.zeros(N, dtype=bool)

    tag = _model_tag(baseline)
    amp_ctx = torch.cuda.amp.autocast(enabled=(device == "cuda"))
    with torch.inference_mode(), amp_ctx:
        for start in tqdm(range(0, len(sids), batch_size), desc=f"{tag}:{word_rep_mode} embed"):
            batch_ids = sids[start : start + batch_size]
            batch_tokens = df_sel.loc[batch_ids, "tokens"].tolist()

            enc_be = tokzr(batch_tokens, **enc_kwargs)
            enc_t = {k: v.to(device) for k, v in enc_be.items()}

            out = model(**enc_t)

            # tuple of (B,T,D); index 0 is embeddings
            hs = out.hidden_states

            for b, sid in enumerate(batch_ids):
                mp: Dict[int, List[int]] = {}
                word_ids = enc_be.word_ids(b)
                for tidx, wid in enumerate(word_ids):
                    if wid is not None:
                        mp.setdefault(int(wid), []).append(int(tidx))

                for gidx, wid in by_sid.get(sid, []):
                    toks = mp.get(wid)
                    if not toks:
                        continue

                    # Build (L,D) by reading each layer tensor at the token positions
                    if word_rep_mode == "first":
                        vec = torch.stack([hs[l][b, toks[0], :] for l in range(L)], dim=0)
                    elif word_rep_mode == "last":
                        vec = torch.stack([hs[l][b, toks[-1], :] for l in range(L)], dim=0)
                    else:  # mean
                        vec = torch.stack([hs[l][b, toks, :].mean(dim=0) for l in range(L)], dim=0)

                    reps[:, gidx, :] = vec.detach().cpu().to(torch.float16).numpy()
                    filled[gidx] = True

            del enc_be, enc_t, out, hs
            if device == "cuda":
                torch.cuda.empty_cache()

    missing = int((~filled).sum())
    if missing:
        print(f"⚠ Missing vectors for {missing} of {N} tokens (dropping).")

    reps = reps[:, filled]

    del model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    return reps, filled


In [ ]:
def _bs_layer_loop(
    rep_sub: np.ndarray,
    M: int,
    n_reps: int,
    compute_once: Callable[[np.ndarray], float],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Bootstrap: sample M with replacement, compute metric per layer.
    rep_sub: (L,N,D)
    """
    L, N, D = rep_sub.shape
    rng = np.random.default_rng(RAND_SEED)
    A = np.full((n_reps, L), np.nan, np.float32)

    for r in tqdm(range(n_reps), desc=f"bootstrap[{compute_once.__name__}]"):
        idx = rng.integers(0, N, size=M)
        for l in range(L):
            X = rep_sub[l, idx].astype(np.float32, copy=False)
            try:
                A[r, l] = float(compute_once(X))
            except Exception:
                A[r, l] = np.nan

    mu = np.nanmean(A, axis=0).astype(np.float32)
    lo = np.nanpercentile(A, 2.5, axis=0).astype(np.float32)
    hi = np.nanpercentile(A, 97.5, axis=0).astype(np.float32)
    return mu, lo, hi

def _compute_metric_bootstrap(rep: np.ndarray, metric: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    N = rep.shape[1]
    if metric in FAST_ONCE:
        M = min(FAST_BS_MAX_SAMP, N)
        n_bs = N_BOOTSTRAP_FAST
        compute_once = FAST_ONCE[metric]
    else:
        compute_once = HEAVY_ONCE.get(metric)
        if compute_once is None:
            return (np.full(rep.shape[0], np.nan),) * 3
        M = min(HEAVY_BS_MAX_SAMP, N)
        n_bs = N_BOOTSTRAP_HEAVY

    return _bs_layer_loop(rep, M, n_bs, compute_once)

def compute_all_metrics_for_rep(rep: np.ndarray) -> Dict[str, Dict[str, np.ndarray]]:
    results: Dict[str, Dict[str, np.ndarray]] = {}
    for metric in ALL_METRICS:
        mu, lo, hi = _compute_metric_bootstrap(rep, metric)
        results[metric] = {"mean": mu, "lo": lo, "hi": hi, "n": int(rep.shape[1])}
    return results


In [ ]:
def save_metric_csv_alltokens(metric: str, results: Dict[tuple, Dict[str, np.ndarray]], layers: np.ndarray):
    rows = []
    for (model, mode), stats in results.items():
        mu, lo, hi = stats["mean"], stats.get("lo"), stats.get("hi")
        tag = _model_tag(model)
        for i, layer_id in enumerate(layers):
            rows.append({
                "model": model,
                "model_tag": tag,
                "word_rep_mode": mode,
                "metric": metric,
                "layer": int(layer_id),
                "mean": float(mu[i]) if np.isfinite(mu[i]) else np.nan,
                "ci_low": float(lo[i]) if isinstance(lo, np.ndarray) and np.isfinite(lo[i]) else np.nan,
                "ci_high": float(hi[i]) if isinstance(hi, np.ndarray) and np.isfinite(hi[i]) else np.nan,
                "n_tokens": int(stats.get("n", 0)),
                "source_csv": Path(CSV_PATH).name,
            })
    df = pd.DataFrame(rows)
    out = CSV_DIR / f"alltokens_{metric}.csv"
    df.to_csv(out, index=False)

def plot_metric_compare(results: Dict[tuple, Dict[str, np.ndarray]], layers: np.ndarray, metric: str):
    plt.figure(figsize=(10, 5.5))
    for (model, mode), stats in results.items():
        mu, lo, hi = stats["mean"], stats.get("lo"), stats.get("hi")
        label = f"{MODEL_LABEL.get(model, model)} • {mode}"
        if mu is None or np.all(np.isnan(mu)):
            continue
        plt.plot(layers, mu, label=label, lw=1.8)
        if isinstance(lo, np.ndarray) and isinstance(hi, np.ndarray) and not np.all(np.isnan(lo)):
            plt.fill_between(layers, lo, hi, alpha=0.15)

    plt.xlabel("Layer")
    plt.ylabel(LABELS.get(metric, metric))
    plt.title(f"{LABELS.get(metric, metric)} • all tokens")
    plt.legend(ncol=2, fontsize="small", frameon=False)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"alltokens_{metric}.png", dpi=220)
    plt.close()


In [ ]:
def run_alltokens_compare():
    if CSV_PATH is None:
        raise ValueError("CSV_PATH is None.")
    if not Path(CSV_PATH).exists():
        raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

    df_all, word_df = load_word_df(CSV_PATH, EXCLUDE_POS)
    all_df = sample_all(word_df, ALL_TOKEN_CAP)
    print(f"✓ pooled tokens: {len(all_df):,}  (cap={ALL_TOKEN_CAP})")

    all_layers = None
    combined: Dict[str, Dict[tuple, Dict[str, np.ndarray]]] = {m: {} for m in ALL_METRICS}

    for model in MODELS:
        for mode in REP_MODES:
            reps, filled = embed_subset(df_all, all_df, model, mode, BATCH_SIZE)

            L = reps.shape[0]
            if all_layers is None:
                all_layers = np.arange(L)

            print(f"→ {model} • {mode}: reps shape = {reps.shape}")

            per_metric = compute_all_metrics_for_rep(reps)
            for metric, stats in per_metric.items():
                combined[metric][(model, mode)] = stats

            del reps
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()

    for metric, results in combined.items():
        save_metric_csv_alltokens(metric, results, all_layers)
        plot_metric_compare(results, all_layers, metric)

    print("✓ done; outputs in:", OUT_DIR.resolve())

# RUN:
run_alltokens_compare()


# 3d PCA

In [ ]:

import os, gc, ast, random, inspect
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel, GPT2TokenizerFast
from sklearn.decomposition import PCA

# Plotly for interactive 3D
import plotly.graph_objects as go

# =============================== CONFIG ===============================
CSV_PATH   = _resolve_csv_path("en_ewt-ud-train_sentences.csv")  # columns: sentence_id (str), tokens (list[str])
MODEL_ID   = "gpt2"                           # e.g., "gpt2", "bert-base-uncased"
REP_MODE   = "last"                           # "first" | "last" | "mean"
BATCH_SIZE = 4
RAND_SEED  = 42

# For plotting (subsample to keep the browser smooth)
PCA_MAX_POINTS = None                         # None = plot all tokens

# Output
OUT_DIR  = _project_out("pca3d_all_tokens"); OUT_DIR.mkdir(parents=True, exist_ok=True)
HTML_OUT = OUT_DIR / f"{MODEL_ID.replace('/','_')}_pca3d_layers.html"

# Repro + device
os.environ["TOKENIZERS_PARALLELISM"] = "true"
random.seed(RAND_SEED); np.random.seed(RAND_SEED); torch.manual_seed(RAND_SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.backends.cudnn.benchmark = True

# =============================== PLOT SETTINGS ===============================
# Bigger fonts: set axes to 17 (what you asked)
AXIS_TITLE_FONT_SIZE = 24
AXIS_TICK_FONT_SIZE  = 20
GLOBAL_FONT_SIZE     = 20
SLIDER_FONT_SIZE     = 20
HOVER_FONT_SIZE      = 20
TITLE_FONT_SIZE      = 20



MARKER_SIZE    = 2
MARKER_OPACITY = 0.70

# ---- Color options (easy to change) ----
# Choose one:
#   COLOR_MODE = "constant"   -> single fixed color for all points
#   COLOR_MODE = "per_layer"  -> different color per layer (edit LAYER_COLORS)
#   COLOR_MODE = "colorscale" -> gradient color per point (by pc1/pc2/pc3/radius)
COLOR_MODE = "constant"

# If COLOR_MODE == "constant"
MARKER_COLOR = "orange"  # any CSS color: "red", "#ff8800", "rgb(255,0,0)", etc.

# If COLOR_MODE == "per_layer"
LAYER_COLORS = None
# Example:
# LAYER_COLORS = ["#636EFA","#EF553B","#00CC96","#AB63FA","#FFA15A","#19D3F3","#FF6692","#B6E880"]

# If COLOR_MODE == "colorscale"
COLOR_BY   = "pc1"       # "pc1" | "pc2" | "pc3" | "radius"
COLOR_SCALE = "Turbo"    # "Viridis", "Cividis", "Plasma", "Inferno", "Turbo", etc.
SHOW_COLORBAR = True

# Optional: show the figure window (useful in notebooks / interactive sessions)
SHOW_FIG = False

# =============================== HELPERS ===============================
def _to_list(x):
    return ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x

def _num_hidden_layers(model) -> int:
    n = getattr(model.config, "num_hidden_layers", None)
    if n is None: n = getattr(model.config, "n_layer", None)
    if n is None: raise ValueError("Cannot determine num_hidden_layers from model.config")
    return int(n)

def _hidden_size(model) -> int:
    d = getattr(model.config, "hidden_size", None)
    if d is None: d = getattr(model.config, "n_embd", None)
    if d is None: raise ValueError("Cannot determine hidden_size from model.config")
    return int(d)

def _load_tok_and_model(model_id: str):
    """
    Robust loader:
      - For GPT-2, force the fast tokenizer and try both 'gpt2' and 'openai-community/gpt2'.
      - Set PAD = EOS for GPT-2-like tokenizers.
    """
    cands = [model_id]
    if "gpt2" in model_id.lower():
        if model_id != "openai-community/gpt2": cands.append("openai-community/gpt2")
        if model_id != "gpt2": cands.append("gpt2")

    last_err = None
    for mid in cands:
        try:
            if "gpt2" in mid.lower():
                tok = GPT2TokenizerFast.from_pretrained(mid, add_prefix_space=True)
            else:
                tok = AutoTokenizer.from_pretrained(mid, use_fast=True, add_prefix_space=True)

            # pad-right and PAD token if missing (common for GPT-2)
            if getattr(tok, "padding_side", None) != "right":
                tok.padding_side = "right"
            if tok.pad_token is None and getattr(tok, "eos_token", None) is not None:
                tok.pad_token = tok.eos_token

            mdl = AutoModel.from_pretrained(mid, output_hidden_states=True)
            if getattr(mdl.config, "pad_token_id", None) is None and tok.pad_token_id is not None:
                mdl.config.pad_token_id = tok.pad_token_id
            if device == "cuda":
                mdl = mdl.half()
            return tok, mdl.eval().to(device), mid
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"Failed to load model/tokenizer for {cands}: {last_err}")

def load_word_df(csv_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (df_all_sentences, token_index_df) for ALL tokens (no labels needed)."""
    df = pd.read_csv(csv_path, usecols=["sentence_id","tokens"])
    df["sentence_id"] = df["sentence_id"].astype(str)
    df.tokens = df.tokens.apply(_to_list)
    rows = []
    for sid, toks in df[["sentence_id","tokens"]].itertuples(index=False):
        for wid in range(len(toks)):
            rows.append((sid, wid))
    word_df = pd.DataFrame(rows, columns=["sentence_id","word_id"])
    return df, word_df

# =============================== EMBEDDING ===============================
def embed_all_tokens(df_all: pd.DataFrame,
                     token_df: pd.DataFrame,
                     model_id: str,
                     rep_mode: str = "mean",
                     batch_size: int = 4):
    """
    Embed all tokens with per-word piece aggregation.
    Returns:
      reps: np.ndarray of shape (L, N, D)
      words: list[str] length N (token strings for hover)
      model_tag: str
    """
    tokzr, model, model_tag = _load_tok_and_model(model_id)

    # Build per-sentence index: sid -> list[(global_idx, word_id)]
    token_df = token_df.copy()
    token_df["sentence_id"] = token_df["sentence_id"].astype(str)
    by_sid: Dict[str, List[Tuple[int,int]]] = {}
    for gidx, (sid, wid) in enumerate(token_df[["sentence_id","word_id"]].itertuples(index=False)):
        by_sid.setdefault(sid, []).append((gidx, int(wid)))

    sids = list(by_sid.keys())
    df_sel = (df_all[df_all.sentence_id.isin(sids)]
              .drop_duplicates("sentence_id")
              .set_index("sentence_id")
              .loc[sids])

    L = _num_hidden_layers(model) + 1
    D = _hidden_size(model)
    N = len(token_df)

    reps   = np.zeros((L, N, D), np.float16)
    words  = [""] * N
    filled = np.zeros(N, dtype=bool)

    enc_kwargs = dict(is_split_into_words=True, return_tensors="pt", padding=True, truncation=True)
    if "add_prefix_space" in inspect.signature(tokzr.__call__).parameters:
        enc_kwargs["add_prefix_space"] = True

    with torch.no_grad(), torch.cuda.amp.autocast(device == "cuda"):
        for start in tqdm(range(0, len(sids), batch_size), desc=f"{model_tag}: embed ({rep_mode})"):
            batch_ids    = sids[start : start + batch_size]
            batch_tokens = df_sel.loc[batch_ids, "tokens"].tolist()

            enc_be = tokzr(batch_tokens, **enc_kwargs)
            enc_t  = {k: v.to(device) for k, v in enc_be.items()}
            out = model(**enc_t)
            h = torch.stack(out.hidden_states).detach().cpu().numpy().astype(np.float32)  # (L,B,T,D)

            for b, sid in enumerate(batch_ids):
                # map word_id -> token positions
                mp: Dict[int, List[int]] = {}
                wids = enc_be.word_ids(b)
                if wids is None:
                    raise RuntimeError("Fast tokenizer required (word_ids() unavailable).")
                for tidx, wid in enumerate(wids):
                    if wid is not None:
                        mp.setdefault(int(wid), []).append(int(tidx))

                toks_for_sent = df_sel.loc[sid, "tokens"]
                for gidx, wid in by_sid.get(sid, []):
                    tokpos = mp.get(wid)
                    if not tokpos:
                        continue

                    # choose representation per wordpiece policy
                    if rep_mode == "first":
                        vec = h[:, b, tokpos[0], :]              # (L, D)
                    elif rep_mode == "last":
                        vec = h[:, b, tokpos[-1], :]             # (L, D)
                    else:  # "mean"
                        vec = h[:, b, tokpos, :].mean(axis=1)    # (L, D)

                    reps[:, gidx, :] = vec.astype(np.float16, copy=False)
                    words[gidx] = str(toks_for_sent[wid])
                    filled[gidx] = True

            del enc_be, enc_t, out, h
            if device == "cuda":
                torch.cuda.empty_cache()

    if (~filled).any():
        missing = int((~filled).sum())
        print(f"⚠ Missing vectors for {missing} tokens (skipped in PCA).")
        reps   = reps[:, filled]
        words  = [w for w, f in zip(words, filled) if f]

    return reps, words, model_tag

# =============================== PCA + PLOTLY ===============================
def _layer_color(l: int) -> str:
    """Pick a per-layer color if LAYER_COLORS is provided; otherwise fall back."""
    if LAYER_COLORS is not None and l < len(LAYER_COLORS):
        return LAYER_COLORS[l]
    return MARKER_COLOR

def _colorscale_values(Y: np.ndarray) -> np.ndarray:
    """Compute numeric values used for colorscale coloring."""
    if COLOR_BY == "pc1":
        return Y[:, 0]
    if COLOR_BY == "pc2":
        return Y[:, 1]
    if COLOR_BY == "pc3":
        return Y[:, 2]
    if COLOR_BY == "radius":
        return np.linalg.norm(Y, axis=1)
    raise ValueError("COLOR_BY must be one of: 'pc1', 'pc2', 'pc3', 'radius'.")

def pca3d_per_layer_and_plot(reps: np.ndarray, words: List[str], model_tag: str):
    """
    reps: (L, N, D), words: list[str] length N
    Creates an interactive 3D Plotly figure with a layer slider.
    """
    L, N, D = reps.shape

    # Optional subsampling for plotting
    if PCA_MAX_POINTS is None or PCA_MAX_POINTS >= N:
        sel_idx = np.arange(N, dtype=np.int64)
    else:
        sel_idx = np.random.default_rng(RAND_SEED).choice(N, size=PCA_MAX_POINTS, replace=False)
        print(f"⚠ PCA_MAX_POINTS={PCA_MAX_POINTS} < N={N} → plotting a subset.")

    reps_sel  = reps[:, sel_idx, :].astype(np.float32, copy=False)
    words_sel = [words[i] for i in sel_idx]

    # PCA to 3D for each layer (independently)
    Y_layers: List[np.ndarray] = []
    for l in range(L):
        X = reps_sel[l]  # (n, D)
        Xc = X - X.mean(0, keepdims=True)
        pca = PCA(n_components=3, random_state=RAND_SEED)
        Y = pca.fit_transform(Xc)  # (n, 3)
        Y_layers.append(Y)

    # Build Plotly figure with a slider to switch layers
    traces = []
    for l in range(L):
        Y = Y_layers[l]

        # Choose marker coloring
        if COLOR_MODE == "constant":
            marker = dict(size=MARKER_SIZE, opacity=MARKER_OPACITY, color=MARKER_COLOR)

        elif COLOR_MODE == "per_layer":
            marker = dict(size=MARKER_SIZE, opacity=MARKER_OPACITY, color=_layer_color(l))

        elif COLOR_MODE == "colorscale":
            cvals = _colorscale_values(Y)
            marker = dict(
                size=MARKER_SIZE,
                opacity=MARKER_OPACITY,
                color=cvals,
                colorscale=COLOR_SCALE,
                showscale=SHOW_COLORBAR,
                colorbar=dict(
                    title=COLOR_BY,
                    titlefont=dict(size=AXIS_TITLE_FONT_SIZE),
                    tickfont=dict(size=AXIS_TICK_FONT_SIZE),
                ),
            )
        else:
            raise ValueError("COLOR_MODE must be one of: 'constant', 'per_layer', 'colorscale'.")

        hovertemplate = (
            "<b>%{text}</b><br>"
            "x=%{x:.3f}<br>y=%{y:.3f}<br>z=%{z:.3f}"
            "<extra>Layer " + str(l) + "</extra>"
        )

        traces.append(
            go.Scatter3d(
                x=Y[:, 0], y=Y[:, 1], z=Y[:, 2],
                mode="markers",
                marker=marker,
                text=words_sel,                 # <-- enables %{text} in hovertemplate
                hovertemplate=hovertemplate,
                name=f"Layer {l}",
                visible=(l == 0),
                showlegend=False,
            )
        )

    steps = []
    for l in range(L):
        vis = [False] * L
        vis[l] = True
        steps.append(dict(
            method="update",
            args=[
                {"visible": vis},
                {"title": {
                    "text": f"{model_tag} • PCA 3D • layer {l} (drag to rotate)",
                    "font": {"size": TITLE_FONT_SIZE},
                }},
            ],
            label=str(l),
        ))

    sliders = [dict(
        active=0,
        steps=steps,
        currentvalue={"prefix": "Layer: ", "font": {"size": SLIDER_FONT_SIZE}},
        font={"size": SLIDER_FONT_SIZE},
        pad={"t": 10}
    )]

    layout = go.Layout(
        title={"text": f"{model_tag} • PCA 3D • layer 0 (drag to rotate)",
               "font": {"size": TITLE_FONT_SIZE}},
        font={"size": GLOBAL_FONT_SIZE},
        hoverlabel={"font": {"size": HOVER_FONT_SIZE}},

        scene=dict(
            xaxis=dict(
                title=dict(text="PC1", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            ),
            yaxis=dict(
                title=dict(text="PC2", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            ),
            zaxis=dict(
                title=dict(text="PC3", font=dict(size=AXIS_TITLE_FONT_SIZE)),
                tickfont=dict(size=AXIS_TICK_FONT_SIZE),
            ),
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=60),
        sliders=sliders,
    )

    fig = go.Figure(data=traces, layout=layout)

    if SHOW_FIG:
        fig.show()

    fig.write_html(str(HTML_OUT), include_plotlyjs="cdn")
    print("✓ Saved interactive HTML to:", HTML_OUT)

# =============================== DRIVER ===============================
def run_pca3d_all_tokens():
    df_all, token_df = load_word_df(CSV_PATH)
    reps, words, tag = embed_all_tokens(
        df_all, token_df, MODEL_ID, rep_mode=REP_MODE, batch_size=BATCH_SIZE
    )
    pca3d_per_layer_and_plot(reps, words, tag)

    # Cleanup
    del reps
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

if __name__ == "__main__":
    run_pca3d_all_tokens()


In [ ]:

import os, gc, ast, random, inspect
from pathlib import Path
from typing import Dict, List, Tuple, Callable, Optional

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel

# =============================== OPTIONAL DEPS ===============================
HAS_DADAPY = False
try:
    from dadapy import Data
    HAS_DADAPY = True
except Exception:
    pass

HAS_SKDIM = False
try:
    from skdim.id import (
        MOM, TLE, CorrInt, FisherS, lPCA,
        MLE, DANCo, ESS, MiND_ML, MADA, KNN
    )
    HAS_SKDIM = True
except Exception:
    pass

try:
    from IsoScore import IsoScore
    _HAS_ISOSCORE = True
except Exception:
    _HAS_ISOSCORE = False
    class _IsoScoreFallback:
        @staticmethod
        def IsoScore(X: np.ndarray) -> float:
            C = np.cov(X.T, ddof=0)
            ev = np.linalg.eigvalsh(C)
            if ev.mean() <= 0 or ev[-1] <= 0:
                return 0.0
            # mean/peak eigenvalue ratio in [0,1]; higher ≈ more isotropic
            return float(np.clip(ev.mean() / ev[-1], 0.0, 1.0))
    IsoScore = _IsoScoreFallback()

# =============================== CONFIG ===============================
CSV_PATH   = _resolve_csv_path("en_ewt-ud-train_sentences.csv")
BASELINE   = "bert-base-uncased"
WORD_REP_MODE = "first"
EXCLUDE_POS = {"X", "SYM", "PART", "INTJ", "NOUN","VERB", "PROPN","ADJ","CCONJ","SCONJ", "NOUN", "PUNCT","PRON", "NUM", "ADP","DET", "AUX"}
RAW_MAX_PER_POS = int(1e12)
# Numerics
EPS = 1e-12

# Bootstrap
N_BOOTSTRAP_FAST   = 50
N_BOOTSTRAP_HEAVY  = 200

FAST_BS_MAX_SAMP_PER_POS  = int(1e12)
HEAVY_BS_MAX_SAMP_PER_POS = 5000

# GRIDE multi-scale max neighbor rank
DADAPY_GRID_RANGE_MAX = 64  # 32–128 typical

# TRUE local PCA settings (kNN neighborhoods)
LPCA_N_NEIGHBORS = 100   # used in fit_pw
LPCA_N_JOBS = 1          # set >1 if you want multiprocessing
LPCA_SMOOTH = False      # optional smoothing inside skdim
LPCA_AGG = "mean"        # "mean" or "median" aggregation over dimension_pw_

RAND_SEED = 42

PLOT_DIR = _project_out("results_POS"); PLOT_DIR.mkdir(exist_ok=True, parents=True)
CSV_DIR  = _project_out("tables_POS") / "pos_bootstrap"; CSV_DIR.mkdir(exist_ok=True, parents=True)

BATCH_SIZE = 1  # increase if GPU allows

# Reproducibility & device
os.environ["TOKENIZERS_PARALLELISM"] = "true"
random.seed(RAND_SEED); np.random.seed(RAND_SEED); torch.manual_seed(RAND_SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.backends.cudnn.benchmark = True

# Style
sns.set_style("darkgrid")
plt.rcParams["figure.dpi"] = 120

# =============================== HELPERS ===============================
def _to_list(x):
    return ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x

def _center(X: np.ndarray) -> np.ndarray:
    return X - X.mean(0, keepdims=True)

def _eigvals_from_X(X: np.ndarray) -> np.ndarray:
    """Eigenvalues of covariance up to a constant via SVD of centered X (descending)."""
    Xc = _center(X.astype(np.float32, copy=False))
    try:
        _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        lam = (S**2).astype(np.float64)
        lam.sort()
        return lam[::-1]
    except Exception:
        return np.array([], dtype=np.float64)

def _jitter_unique(X: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Add tiny noise if there are duplicate rows (helps NN-based estimators)."""
    try:
        if np.unique(X, axis=0).shape[0] < X.shape[0]:
            X = X + np.random.normal(scale=eps, size=X.shape).astype(X.dtype)
    except Exception:
        pass
    return X

# =============================== METRICS (ONE SHOT) ===============================
# --- Isotropy (fast) ---

def _fast_isoscore(X: np.ndarray) -> float:
    """Fast IsoScore from covariance eigenvalues; matches the package algorithm."""
    try:
        lam = _eigvals_from_X(X)
    except Exception:
        Xc = np.asarray(X, dtype=np.float32)
        if Xc.ndim != 2 or Xc.size == 0:
            return float("nan")
        Xc = Xc - Xc.mean(0, keepdims=True)
        try:
            _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        except Exception:
            return float("nan")
        lam = (S.astype(np.float64) ** 2)
        lam = np.sort(np.maximum(lam, 0.0))[::-1]

    lam = np.asarray(lam, dtype=np.float64)
    lam = lam[np.isfinite(lam) & (lam >= 0)]
    n = int(lam.size)
    if n < 2:
        return float("nan")
    norm = float(np.linalg.norm(lam))
    if norm <= 0:
        return 0.0

    root_n = np.sqrt(n)
    denom = np.sqrt(2.0 * (n - root_n))
    if denom <= 0:
        return float("nan")

    delta = float(np.linalg.norm((root_n * lam) / norm - 1.0) / denom)
    delta = float(np.clip(delta, 0.0, 1.0))
    phi = (n - (delta ** 2) * (n - root_n)) ** 2 / (n ** 2)
    iso = (n * phi - 1.0) / (n - 1.0)
    return float(np.clip(iso, 0.0, 1.0))


def _iso_once(X: np.ndarray) -> float:
    return float(_fast_isoscore(X))

def _spect_once(X: np.ndarray) -> float:
    ev = np.linalg.eigvalsh(np.cov(X.T, ddof=0))
    return float(ev[-1] / (ev.mean() + 1e-9))

def _rand_once(X: np.ndarray, K: int = 2000) -> float:
    n = X.shape[0]
    if n < 2:
        return np.nan
    rng = np.random.default_rng()
    K_eff = min(K, (n*(n-1))//2)
    i = rng.integers(0, n, size=K_eff)
    j = rng.integers(0, n, size=K_eff)
    same = i == j
    if same.any():
        j[same] = rng.integers(0, n, size=same.sum())
    A, B = X[i], X[j]
    num = np.sum(A*B, axis=1)
    den = (np.linalg.norm(A, axis=1)*np.linalg.norm(B, axis=1) + 1e-9)
    return float(np.mean(np.abs(num/den)))

def _sf_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return np.nan
    gm = np.exp(np.mean(np.log(lam + EPS)))
    am = float(lam.mean() + EPS)
    return float(gm / am)

def _vmf_kappa_once(X: np.ndarray) -> float:
    if X.shape[0] < 2:
        return np.nan
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    R = np.linalg.norm(Xn.mean(axis=0))
    d = Xn.shape[1]
    if R < 1e-9:
        return 0.0
    # standard closed-form approximation
    return float(max(R * (d - R**2) / (1.0 - R**2 + 1e-9), 0.0))

# --- Linear ID (global spectral, fast) ---
def _pcaXX_once(X: np.ndarray, var_ratio: float) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return np.nan
    c = np.cumsum(lam)
    thr = c[-1] * var_ratio
    return float(np.searchsorted(c, thr) + 1)

def _erank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return np.nan
    p = lam / (lam.sum() + EPS)
    H = -(p * np.log(p + EPS)).sum()
    return float(np.exp(H))

def _pr_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return np.nan
    s1 = lam.sum(); s2 = (lam**2).sum()
    return float((s1**2) / (s2 + EPS))

def _stable_rank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return np.nan
    return float(lam.sum() / (lam.max() + EPS))

# --- Non-linear (heavy) ---
def _dadapy_twonn_once(X: np.ndarray) -> float:
    if not HAS_DADAPY:
        return np.nan
    d = Data(coordinates=_jitter_unique(X))
    id_est, _, _ = d.compute_id_2NN()
    return float(id_est)

def _dadapy_gride_once(X: np.ndarray) -> float:
    if not HAS_DADAPY:
        return np.nan
    d = Data(coordinates=_jitter_unique(X))
    range_max = min(int(DADAPY_GRID_RANGE_MAX), X.shape[0] - 1)
    if range_max < 2:
        return float("nan")
    d.compute_distances(maxk=range_max)
    ids, _, _ = d.return_id_scaling_gride(range_max=range_max)
    return float(ids[-1])

# =============================== skdim WRAPPERS ===============================
def _skdim_factory(name: str):
    """Return a factory that builds a fresh skdim estimator each call, or None."""
    if not HAS_SKDIM:
        return None

    mapping = {
        "mom": MOM, "tle": TLE, "corrint": CorrInt, "fishers": FisherS,
        "mle": MLE, "danco": DANCo, "mind_ml": MiND_ML,
        "mada": MADA, "knn": KNN,
        # local PCA variants (same class, different constructor args)
        "lpca": lPCA,
        "lpca95": lPCA,
        "lpca99": lPCA,
    }

    cls = mapping.get(name)
    if cls is None:
        return None

    def _builder():
        # IMPORTANT: set verbose=False to avoid spam
        if name == "lpca":       # FO variant (Fukunaga-Olsen)
            return cls(ver="FO", verbose=False)
        if name == "lpca95":     # variance-ratio local PCA
            return cls(ver="ratio", alphaRatio=0.95, verbose=False)
        if name == "lpca99":     # variance-ratio local PCA
            return cls(ver="ratio", alphaRatio=0.99, verbose=False)
        return cls()  # default
    return _builder

def _skdim_once_builder(name: str) -> Optional[Callable[[np.ndarray], float]]:
    """
    Build a per-subsample function X->float.

    For lPCA variants we compute *true local PCA*:
      - run fit_pw(...) to get pointwise estimates dimension_pw_
      - aggregate (mean/median) to return one scalar

    For other skdim estimators we keep the library's global fit(...) -> dimension_ behavior.
    """
    build = _skdim_factory(name)
    if build is None:
        return None

    def _once(X: np.ndarray) -> float:
        Xj = _jitter_unique(X)
        est = build()

        # TRUE local PCA: pointwise in kNN, then aggregate
        if name in {"lpca", "lpca95", "lpca99"}:
            n = Xj.shape[0]
            # need at least 3 points and at least 2 neighbors
            if n < 3:
                return np.nan
            k = min(LPCA_N_NEIGHBORS, n - 1)
            if k < 2:
                return np.nan
            # fit_pw creates est.dimension_pw_ (pointwise IDs) :contentReference[oaicite:2]{index=2}
            est.fit_pw(Xj, n_neighbors=k, n_jobs=LPCA_N_JOBS, smooth=LPCA_SMOOTH)
            dims = getattr(est, "dimension_pw_", None)
            if dims is None:
                return np.nan
            dims = np.asarray(dims, dtype=np.float64)
            dims = dims[np.isfinite(dims)]
            if dims.size == 0:
                return np.nan
            if LPCA_AGG.lower() == "median":
                return float(np.nanmedian(dims))
            return float(np.nanmean(dims))

        # Other estimators: dataset-level number
        est.fit(Xj)
        return float(getattr(est, "dimension_", np.nan))

    return _once

# =============================== DATA ===============================
def load_word_df(csv_path: str, exclude_pos: set[str] = EXCLUDE_POS):
    df = pd.read_csv(csv_path, usecols=["sentence_id", "tokens", "pos"])
    df["sentence_id"] = df["sentence_id"].astype(str)
    df.tokens = df.tokens.apply(_to_list)
    df.pos = df.pos.apply(_to_list)

    rows = []
    for sid, toks, poss in df[["sentence_id", "tokens", "pos"]].itertuples(index=False):
        for wid, (tok, p) in enumerate(zip(toks, poss)):
            if p not in exclude_pos:
                rows.append((sid, wid, p, tok))
    word_df = pd.DataFrame(rows, columns=["sentence_id", "word_id", "pos", "word"])
    return df, word_df

def sample_raw(word_df: pd.DataFrame, per_pos_cap: int = RAW_MAX_PER_POS) -> pd.DataFrame:
    """Per-POS cap without frequency matching."""
    picks = []
    for p, sub in word_df.groupby("pos", sort=False):
        n = min(len(sub), per_pos_cap)
        picks.append(sub.sample(n, random_state=RAND_SEED, replace=False))
    return pd.concat(picks, ignore_index=True)

def make_class_palette(classes: List[str]) -> Dict[str, Tuple[float, float, float]]:
    """Deterministic {class -> RGB} palette."""
    base_colors: List[Tuple[float, float, float]] = []
    for name in ("tab20", "tab20b", "tab20c"):
        try:
            base_colors.extend(sns.color_palette(name, 20))
        except Exception:
            pass

    if len(base_colors) < len(classes):
        base_colors = list(sns.color_palette("husl", len(classes)))

    ordered = list(sorted(classes))
    return {cls: base_colors[i % len(base_colors)] for i, cls in enumerate(ordered)}

# =============================== EMBEDDING ===============================
def embed_subset(df_all_sentences: pd.DataFrame,
                 subset_df: pd.DataFrame,
                 baseline: str = BASELINE,
                 word_rep_mode: str = WORD_REP_MODE,
                 batch_size: int = BATCH_SIZE) -> Tuple[np.ndarray, np.ndarray]:
    """Return reps (L,N,D) and filled mask (N,) for the selected tokens."""
    df_all_sentences["sentence_id"] = df_all_sentences["sentence_id"].astype(str)
    subset_df["sentence_id"] = subset_df["sentence_id"].astype(str)

    # sid -> list[(global_idx, word_id)]
    by_sid: Dict[str, List[Tuple[int, int]]] = {}
    for gidx, (sid, wid) in enumerate(subset_df[["sentence_id", "word_id"]].itertuples(index=False)):
        by_sid.setdefault(str(sid), []).append((gidx, int(wid)))

    sids = list(by_sid.keys())
    df_sel = (df_all_sentences[df_all_sentences.sentence_id.isin(sids)]
              .drop_duplicates("sentence_id")
              .set_index("sentence_id")
              .loc[sids])

    tokzr = AutoTokenizer.from_pretrained(baseline, use_fast=True)
    enc_kwargs = dict(is_split_into_words=True, return_tensors="pt", padding=True)
    if "add_prefix_space" in inspect.signature(tokzr.__call__).parameters:
        enc_kwargs["add_prefix_space"] = True

    model = AutoModel.from_pretrained(baseline, output_hidden_states=True).eval().to(device)
    if device == "cuda":
        model.half()

    L = model.config.num_hidden_layers + 1
    D = model.config.hidden_size
    N = len(subset_df)

    reps = np.zeros((L, N, D), np.float16)
    filled = np.zeros(N, dtype=bool)

    with torch.no_grad(), torch.cuda.amp.autocast(enabled=(device == "cuda")):
        for start in tqdm(range(0, len(sids), batch_size), desc=f"{baseline} (embed subset)"):
            batch_ids = sids[start: start + batch_size]
            batch_tokens = df_sel.loc[batch_ids, "tokens"].tolist()

            enc_be = tokzr(batch_tokens, **enc_kwargs)
            enc_t = {k: v.to(device) for k, v in enc_be.items()}
            out = model(**enc_t)
            h = torch.stack(out.hidden_states).detach().cpu().numpy().astype(np.float32)  # (L,B,T,D)

            for b, sid in enumerate(batch_ids):
                mp = {}
                for tidx, wid in enumerate(enc_be.word_ids(b)):
                    if wid is not None:
                        mp.setdefault(int(wid), []).append(int(tidx))

                for gidx, wid in by_sid.get(sid, []):
                    toks = mp.get(wid)
                    if not toks:
                        continue
                    if word_rep_mode == "first":
                        vec = h[:, b, toks[0], :]
                    else:
                        vec = h[:, b, toks, :].mean(axis=1)
                    reps[:, gidx, :] = vec.astype(np.float16, copy=False)
                    filled[gidx] = True

            # free batch buffers
            del enc_be, enc_t, out, h
            if device == "cuda":
                torch.cuda.empty_cache()

    missing = int((~filled).sum())
    if missing:
        print(f"⚠ Missing vectors for {missing} of {N} sampled words")

    del model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    return reps, filled

# =============================== BOOTSTRAP CORE ===============================
def _bs_layer_loop(rep_sub: np.ndarray, M: int, n_reps: int, compute_once: Callable[[np.ndarray], float]):
    """Bootstrap: sample M with replacement and apply compute_once(X_layer) -> scalar for each layer."""
    L, N, D = rep_sub.shape
    rng = np.random.default_rng(RAND_SEED)
    A = np.full((n_reps, L), np.nan, np.float32)

    for r in range(n_reps):
        idx = rng.integers(0, N, size=M)
        for l in range(L):
            X = rep_sub[l, idx].astype(np.float32, copy=False)
            try:
                A[r, l] = float(compute_once(X))
            except Exception:
                A[r, l] = np.nan

    mu = np.nanmean(A, axis=0).astype(np.float32)
    lo = np.nanpercentile(A, 2.5, axis=0).astype(np.float32)
    hi = np.nanpercentile(A, 97.5, axis=0).astype(np.float32)
    return mu, lo, hi

# =============================== METRIC REGISTRY ===============================
FAST_ONCE: Dict[str, Callable[[np.ndarray], float]] = {
    # Isotropy
    "iso": _iso_once,
    "spect": _spect_once,
    "rand": _rand_once,
    "sf": _sf_once,
    "vmf_kappa": _vmf_kappa_once,
    # Linear (global spectral)
    "erank": _erank_once,
    "pr": _pr_once,
    "stable_rank": _stable_rank_once,
    # global PCA counts if you ever want them:
    # "pca95": lambda X: _pcaXX_once(X, 0.95),
    # "pca99": lambda X: _pcaXX_once(X, 0.99),
}

HEAVY_ONCE: Dict[str, Optional[Callable[[np.ndarray], float]]] = {
    "twonn": _dadapy_twonn_once,
    "gride": _dadapy_gride_once,

    "mom": _skdim_once_builder("mom"),
    "tle": _skdim_once_builder("tle"),
    "corrint": _skdim_once_builder("corrint"),
    "fishers": _skdim_once_builder("fishers"),
    "mle": _skdim_once_builder("mle"),
    "mada": _skdim_once_builder("mada"),
    "knn": _skdim_once_builder("knn"),

    # TRUE local PCA variants (fit_pw + aggregate)
    "lpca": _skdim_once_builder("lpca"),
    "lpca95": _skdim_once_builder("lpca95"),
    "lpca99": _skdim_once_builder("lpca99"),
}

LABELS = {
    # Isotropy
    "iso": "IsoScore",
    "spect": "Spectral Ratio",
    "rand": "RandCos |μ|",
    "sf": "Spectral Flatness",
    "vmf_kappa": "vMF κ",

    # Linear ID (global spectral)
    "erank": "Effective Rank",
    "pr": "Participation Ratio",
    "stable_rank": "Stable Rank",

    # Linear ID (true local PCA)
    "lpca": "lPCA FO (local)",
    "lpca95": "lPCA 0.95 (local)",
    "lpca99": "lPCA 0.99 (local)",

    # Nonlinear
    "twonn": "TwoNN ID",
    "gride": "GRIDE",
    "mom": "MOM",
    "tle": "TLE",
    "corrint": "CorrInt",
    "fishers": "FisherS",
    "mle": "MLE",
    "mada": "MADA",
    "knn": "KNN",
}

PLOT_ORDER = (
    "iso", "sf", "vmf_kappa", "spect", "rand",
    "erank", "pr", "stable_rank", "lpca95", "lpca99", "lpca",
    "twonn", "gride", "mom", "tle", "corrint", "fishers", "mle", "mada", "knn"
)

# Choose metrics to compute (edit freely)
ALL_METRICS = [ "lpca99"]  # example

# =============================== SAVE / PLOT ===============================
def save_metric_csv_all_pos(metric: str,
                            pos_to_stats: Dict[str, Dict[str, np.ndarray]],
                            layers: np.ndarray,
                            baseline: str,
                            subset_name: str = "raw"):
    rows = []
    for p, stats in pos_to_stats.items():
        mu, lo, hi = stats["mean"], stats.get("lo"), stats.get("hi")
        for l, val in enumerate(mu):
            rows.append({
                "subset": subset_name, "model": baseline, "feature": "pos",
                "class": p, "metric": metric, "layer": int(layers[l]),
                "mean": float(val) if np.isfinite(val) else np.nan,
                "ci_low": float(lo[l]) if isinstance(lo, np.ndarray) and np.isfinite(lo[l]) else np.nan,
                "ci_high": float(hi[l]) if isinstance(hi, np.ndarray) and np.isfinite(hi[l]) else np.nan,
                "n_tokens": int(stats.get("n", 0)), "word_rep_mode": WORD_REP_MODE,
                "source_csv": Path(CSV_PATH).name,
            })
    df = pd.DataFrame(rows)
    out = CSV_DIR / f"pos_{subset_name}_{metric}_{baseline}.csv"
    df.to_csv(out, index=False)
    return out

def plot_metric_with_ci(pos_to_stats: Dict[str, Dict[str, np.ndarray]],
                        layers: np.ndarray, metric: str, title: str, out_path: Path,
                        palette: Dict[str, Tuple[float, float, float]] | None = None):
    plt.figure(figsize=(9, 5))
    for p, stats in pos_to_stats.items():
        mu, lo, hi = stats["mean"], stats.get("lo"), stats.get("hi")
        if mu is None or np.all(np.isnan(mu)):
            continue
        color = palette.get(p) if isinstance(palette, dict) else None
        plt.plot(layers, mu, label=p, lw=1.8, color=color)
        if isinstance(lo, np.ndarray) and isinstance(hi, np.ndarray) and not np.all(np.isnan(lo)):
            plt.fill_between(layers, lo, hi, alpha=0.15, color=color)

    plt.xlabel("Layer")
    plt.ylabel(LABELS.get(metric, metric.upper()))
    plt.title(title)

    n_classes = len(pos_to_stats)
    ncol = 3 if n_classes > 12 else 2
    plt.legend(ncol=ncol, fontsize="small", title="POS", frameon=False)

    plt.tight_layout()
    plt.savefig(out_path, dpi=220)
    plt.close()

# =============================== DRIVER ===============================
def run_pos_pipeline():
    # 1) Load
    df_all, word_df = load_word_df(CSV_PATH, EXCLUDE_POS)
    POS_TAGS = sorted(word_df.pos.unique())
    palette = make_class_palette(POS_TAGS)

    print(f"✓ corpus ready — {len(word_df):,} tokens across {len(POS_TAGS)} POS")
    print(f"• DADApy: {'available' if HAS_DADAPY else 'missing'}  • scikit-dimension: {'available' if HAS_SKDIM else 'missing'}")

    # 2) Raw sampling only
    raw_df = sample_raw(word_df, RAW_MAX_PER_POS)
    print("Sample sizes per POS (raw cap):")
    print(raw_df.pos.value_counts().to_dict())

    # 3) Embed once
    reps, filled = embed_subset(df_all, raw_df, BASELINE, WORD_REP_MODE, BATCH_SIZE)
    raw_df = raw_df.reset_index(drop=True).loc[filled].reset_index(drop=True)
    pos_arr = raw_df.pos.values
    L = reps.shape[0]
    layers = np.arange(L)
    print(f"✓ embedded {len(raw_df):,} tokens  • layers={L}")

    # 4) Metric-by-metric loop
    for metric in ALL_METRICS:
        print(f"\n→ Computing metric: {metric} …")

        if metric in FAST_ONCE:
            compute_once = FAST_ONCE[metric]
            n_bs = N_BOOTSTRAP_FAST
            Mcap = FAST_BS_MAX_SAMP_PER_POS
        else:
            compute_once = HEAVY_ONCE.get(metric)
            n_bs = N_BOOTSTRAP_HEAVY
            Mcap = HEAVY_BS_MAX_SAMP_PER_POS

        if compute_once is None:
            print(f"  (skipping {metric}: estimator unavailable)")
            continue

        metric_results: Dict[str, Dict[str, np.ndarray]] = {}
        for p in POS_TAGS:
            idx = np.where(pos_arr == p)[0]
            if idx.size < 3:
                continue
            sub = reps[:, idx]  # (L, n_p, D)
            Np = sub.shape[1]
            M = min(Mcap, Np)

            mu, lo, hi = _bs_layer_loop(sub, M, n_bs, compute_once)
            metric_results[p] = {"mean": mu, "lo": lo, "hi": hi, "n": int(Np)}

        csv_out = save_metric_csv_all_pos(metric, metric_results, layers, BASELINE, subset_name="raw")
        plot_out = PLOT_DIR / f"raw_{metric}_{BASELINE}.png"
        plot_metric_with_ci(metric_results, layers, metric,
                            title=f"{LABELS.get(metric, metric.upper())} • {BASELINE}",
                            out_path=plot_out,
                            palette=palette)

        print(f"  ✓ saved: CSV={csv_out}  plot={plot_out}")

        del metric_results
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    del reps
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()
    print("\n✓ done (incremental outputs produced per metric).")

if __name__ == "__main__":
    run_pos_pipeline()
